# Parsing the Black Baptist Church Directory (Volume II)

**Author:** Ziqian (Leah) Liu

**Source:** *Directory of Negro Baptist Churches in the United States* (1942), Volume II, digitized by HathiTrust.

**Goal:** Parse the OCR text into a structured dataset with one row per church and the following columns:
1. **Church name**
2. **Church city**
3. **Pastor name**
4. **Pastor address**
5. **State**
6. **Association**

**Pipeline:**
- **Step 1:** Load and clean raw text (remove page markers, page numbers, noise)
- **Step 2:** Identify and remove non-entry lines (state headers, association blocks, officer lines)
- **Step 3:** Group remaining lines into church entries (detect church-name lines, group continuation lines)
- **Step 4:** Parse each entry into structured columns
- **Step 5:** Geocode churches (Census → Nominatim), recover unmatched entries, and match points to 1980 SMSAs

**Volume II states:** Mississippi, Missouri, Montana, Nebraska, New Jersey, New York, North Carolina, Ohio, Oklahoma, South Carolina, Tennessee, Texas, Virginia, Washington, Washington D.C., West Virginia, Pennsylvania (Appendix)

## Step 1: Load and Clean Raw Text

Remove formatting artifacts inserted by HathiTrust digitization and OCR: page boundary markers, printed page numbers, and decorative noise.

### 1.1 Load the file

The input is a plain-text file extracted from HathiTrust's digitized scan. Each page is separated by a marker like `## p. 10 (#18) ###...`.

In [1]:
import re
import pandas as pd
from collections import Counter

# --- File path ---
INPUT_PATH = (
    "./"
    "1.data/2.raw/Covariates/"
    "Black Churches/Volume II/"
    "mdp-39015024219845-1782143319.txt"
)

# --- Load raw text ---
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

lines = raw_text.split("\n")
print(f"Total lines in raw file: {len(lines):,}")

Total lines in raw file: 27,079


### 1.2 Extract content pages

Church entries run from printed page 3 (absolute page #11, first Mississippi entries) through the end of the Pennsylvania Appendix (page 314). The Association Index starts at page 317 (#327) and the Pennsylvania municipal index starts at page 315 (#325) — both are excluded.

In [2]:
start_marker = "## p. 3 (#11)"    # first page with church entries (Mississippi)
end_marker   = "## p. 315 (#325)" # Pennsylvania municipal index starts here

start_idx = None
end_idx   = None

for i, line in enumerate(lines):
    if start_marker in line and start_idx is None:
        start_idx = i
    if end_marker in line:
        end_idx = i
        break

print(f"Content range: line {start_idx} -> {end_idx}")

content_lines = lines[start_idx:end_idx]
print(f"Extracted {len(content_lines):,} lines")

Content range: line 237 -> 22775
Extracted 22,538 lines


### 1.3 Remove page markers

HathiTrust inserts markers like `## p. 10 (#18) ###############...` between every page.

In [3]:
page_marker_re = re.compile(r"^## p\.")

before = len(content_lines)
content_lines = [l for l in content_lines if not page_marker_re.match(l)]
print(f"Removed {before - len(content_lines)} page markers")
print(f"{len(content_lines):,} lines remain")

Removed 314 page markers
22,224 lines remain


### 1.4 Remove page number lines and decorative noise

Printed page numbers appear in various OCR formats: `- 4 -`, `- 10 -`, `3`, etc.

In [4]:
# Page numbers: optional punctuation around a 1-3 digit number
page_num_re = re.compile(r"^\s*[-\xb7.]*\s*\d{1,3}\s*[-\xb7.]*\s*$")

page_nums_found = [l for l in content_lines if page_num_re.match(l)]
content_lines   = [l for l in content_lines if not page_num_re.match(l)]
print(f"Removed {len(page_nums_found)} page-number lines. Examples:")
for l in page_nums_found[:8]:
    print(f"  '{l.strip()}'")

# Decorative noise: lines that are only dashes, dots, middle-dots, etc.
noise_re = re.compile(r"^\s*[-\xb7.*+]+\s*$")

noise_found   = [l for l in content_lines if noise_re.match(l)]
content_lines = [l for l in content_lines if not noise_re.match(l)]
print(f"Removed {len(noise_found)} decorative-noise lines")

Removed 334 page-number lines. Examples:
  '3'
  '- 4 -'
  '- 5 -'
  '- 6 -'
  '1'
  '8'
  '9'
  '10-'
Removed 304 decorative-noise lines


### 1.5 Remove blank lines, strip whitespace, and preview

In [5]:
content_lines = [l.strip() for l in content_lines if l.strip()]

sep = "=" * 65
print(f"After all Step 1 cleaning: {len(content_lines):,} lines")
print(f"\n{sep}")
print("Preview -- first 40 lines after Step 1:")
print(sep)
for i, l in enumerate(content_lines[:40]):
    print(f"  {i:>5}: {l}")

After all Step 1 cleaning: 20,644 lines

Preview -- first 40 lines after Step 1:
      0: Adams County Missionary
      1: Baptist Association
      2: MISSISSIPPI
      3: Mississippi
      4: General Missionary Baptist State Convention
      5: Rev. A. A. Cosey, President
      6: Vicksburg
      7: Rev. A. W. Moore, Corresponding Secretary
      8: 107 E. Percy St., Greenwood
      9: Adams County Missionary Baptist Association
     10: Rev. C. R. Anderson, Moderator
     11: 10 Keim Ave., Natchez
     12: Antioch Baptist Church
     13: Natchez; Rev. C. R. Anderson
     14: 10 Keim Ave., Natchez
     15: Beulah Baptist Church
     16: Natchez; Rev. B. D. Sims
     17: Bright Morning Star Baptist Church
     18: Natchez; Rev. R. A. Mays
     19: China Grove Baptist Church
     20: Natchez; Rev. C. R. Anderson
     21: 10 Keim Ave., Natchez
     22: Clay Mount Baptist Church
     23: Natchez; Rev. J. A. Briscoe
     24: Ebenezer Baptist Church
     25: Natchez; Rev. C. Ellis
     26:

## Step 2: Identify and Remove Non-Entry Lines

After Step 1, the text still contains non-entry content mixed in with church entries:

- **State headers:** standalone lines like "Mississippi", "Ohio"
- **Association / Convention declarations:** multi-line blocks naming the association and its officers
- **Running headers:** association names repeated at the top of pages
- **Association name fragments:** partial names like "Bolivar County Missionary" that span two lines
- **Officer lines:** stray moderator/secretary/president lines

### 2.1 Define reference lists

US state names for identifying state header lines, and keywords for identifying association name fragments.

In [6]:
# US states that appear in this volume (Volume II)
US_STATES_LOWER = {
    "mississippi", "missouri", "montana", "nebraska",
    "new jersey", "new york", "north carolina", "ohio",
    "oklahoma", "south carolina", "tennessee", "texas",
    "virginia", "washington", "west virginia", "pennsylvania",
    "d. c.",
}

# Keywords typical of association/convention names but almost never in
# church entry lines.
ASSOC_FRAGMENT_KEYWORDS = {"District", "County", "Missionary", "Colored", "State"}

### 2.2 Classify and remove non-entry lines

Rules (applied in order for each line):

| Rule | Trigger | Action |
|------|---------|--------|
| 1 | Line matches a US state name | Remove as state header; update current state |
| 2 | Line has Association/Convention keyword + officers follow within 5 lines | Remove entire block (declaration + officers) |
| 3 | Line has Association/Convention keyword but no officers follow | Remove as running header |
| 4 | Line has fragment keyword (District, County, etc.) but no Church/Rev. | Remove as association fragment |
| 5 | Line contains officer title (Moderator, Secretary, etc.) | Remove as stray officer |
| 6 | None of the above | Keep as church entry content |

In [7]:
# OCR-robust regex patterns used throughout Step 2
ASSOC_RE = r"\b(Associations?|Asspciation|Convention)\b|\bAssociati\s+on\b"
OFFICER_RE = r"\b(Mod[eo]rat\s*or|Hoderat\s*or|Secretary|Socrotary|Secrteary|President|Treasurer)\b"
CHURCH_RE = r"\bChur\s*ch\b|\b(Churoh|Chruch|Churah|Churchi|Chapel|Chapol|Temple|Tabernacle|Mission)\b|@hurch"
REV_RE = r"^(Rev\.|Kev\.|Rov\.)"
REV_ANYWHERE_RE = r"\bRev\.|Kev\.|Rov\."

entries = []    # lines kept: church entry content
removed = []    # lines removed: headers, officers, etc.

current_state = None
current_association = None

i = 0
while i < len(content_lines):
    line = content_lines[i]

    # ---- Rule 1: State header ----
    if line.lower() in US_STATES_LOWER:
        current_state = line.title()
        removed.append({"text": line, "type": "state_header"})
        i += 1
        continue

    # Precompute flags used by multiple rules
    has_assoc  = bool(re.search(ASSOC_RE, line))
    has_church = bool(re.search(CHURCH_RE, line))
    starts_rev = bool(re.match(REV_RE, line))

    # ---- Rule 2 & 3: Association / Convention lines ----
    if has_assoc and not has_church and not starts_rev:

        # Look ahead up to 5 lines for officer keywords
        has_officers = False
        for j in range(i + 1, min(i + 6, len(content_lines))):
            if re.search(OFFICER_RE, content_lines[j]):
                has_officers = True
                break

        if has_officers:
            # --- Rule 2: Full declaration block ---
            if len(line) > 15:
                current_association = line
            removed.append({"text": line, "type": "assoc_declaration"})
            i += 1

            # Gobble subsequent lines until we hit entry content
            while i < len(content_lines):
                next_line = content_lines[i]

                # Stop if we reach actual church-entry content
                is_entry_content = (
                    bool(re.search(CHURCH_RE, next_line))
                    or (
                        re.match(REV_RE, next_line)
                        and not re.search(OFFICER_RE, next_line)
                    )
                )
                if is_entry_content:
                    break

                # Update metadata if we encounter state/assoc names in the block
                if next_line.lower() in US_STATES_LOWER:
                    current_state = next_line.title()
                if (re.search(ASSOC_RE, next_line)
                        and len(next_line) > 15):
                    current_association = next_line

                # Classify the removed line (for reporting)
                if re.search(OFFICER_RE, next_line):
                    removed.append({"text": next_line, "type": "officer_line"})
                elif next_line.lower() in US_STATES_LOWER:
                    removed.append({"text": next_line, "type": "state_in_block"})
                elif re.search(ASSOC_RE, next_line):
                    removed.append({"text": next_line, "type": "assoc_continuation"})
                else:
                    removed.append({"text": next_line, "type": "officer_address"})
                i += 1
            continue

        else:
            # --- Rule 3: Running header (no officers follow) ---
            removed.append({"text": line, "type": "assoc_running_header"})
            i += 1
            continue

    # ---- Rule 4: Association name fragment ----
    has_fragment_kw = any(kw in line for kw in ASSOC_FRAGMENT_KEYWORDS)
    has_rev_anywhere = bool(re.search(REV_ANYWHERE_RE, line))
    if has_fragment_kw and not has_church and not starts_rev and not has_rev_anywhere:
        removed.append({"text": line, "type": "assoc_fragment"})
        i += 1
        continue

    # ---- Rule 5: Stray officer line ----
    if re.search(OFFICER_RE, line):
        removed.append({"text": line, "type": "stray_officer"})
        i += 1
        continue

    # ---- Rule 6: Keep as church entry content ----
    entries.append({
        "text": line,
        "state": current_state,
        "association": current_association,
    })
    i += 1

print(f"Lines kept  (entries): {len(entries):,}")
print(f"Lines removed:         {len(removed):,}")

Lines kept  (entries): 18,498
Lines removed:         2,146


### 2.3 Preview results and verify

Two things to check:
1. Were real church entries accidentally removed?
2. Were non-entry lines accidentally kept?

In [8]:
sep = "=" * 70

# --- Breakdown of removed lines ---
print("Breakdown of removed lines by type:")
for rtype, count in Counter(r["type"] for r in removed).most_common():
    print(f"  {rtype:30s}: {count:>5}")

print(f"\n{sep}")
print("REMOVED LINES -- sample by type (check for false positives)")
print(sep)

for rtype in sorted(set(r["type"] for r in removed)):
    sample = [r for r in removed if r["type"] == rtype][:5]
    print(f"\n  [{rtype}]")
    for r in sample:
        print(f"    \"{r['text']}\"")

Breakdown of removed lines by type:
  officer_address               :   513
  officer_line                  :   449
  state_header                  :   268
  assoc_fragment                :   251
  assoc_declaration             :   247
  assoc_running_header          :   236
  assoc_continuation            :   107
  state_in_block                :    49
  stray_officer                 :    26

REMOVED LINES -- sample by type (check for false positives)

  [assoc_continuation]
    "General Missionary Baptist State Convention"
    "Adams County Missionary Baptist Association"
    "Hinds County District Association"
    "Jackson Missionary Baptist District Association"
    "Jefferson County Association"

  [assoc_declaration]
    "Baptist Association"
    "African Missionary Baptist Association of Wilkinson County"
    "Bolivar County Missionary Baptist Association"
    "Claiborne County District Association"
    "Coahoma County District Association"

  [assoc_fragment]
    "Adams County 

In [9]:
sep = "=" * 70
print(sep)
print("KEPT LINES -- verify these look like church entry content")
print(sep)
for e in entries[:50]:
    st    = e['state'] or '???'
    assoc = (e['association'] or '???')[:40]
    print(f"  [{st:20s} | {assoc:40s}]  {e['text']}")

KEPT LINES -- verify these look like church entry content
  [Mississippi          | Adams County Missionary Baptist Associat]  Antioch Baptist Church
  [Mississippi          | Adams County Missionary Baptist Associat]  Natchez; Rev. C. R. Anderson
  [Mississippi          | Adams County Missionary Baptist Associat]  10 Keim Ave., Natchez
  [Mississippi          | Adams County Missionary Baptist Associat]  Beulah Baptist Church
  [Mississippi          | Adams County Missionary Baptist Associat]  Natchez; Rev. B. D. Sims
  [Mississippi          | Adams County Missionary Baptist Associat]  Bright Morning Star Baptist Church
  [Mississippi          | Adams County Missionary Baptist Associat]  Natchez; Rev. R. A. Mays
  [Mississippi          | Adams County Missionary Baptist Associat]  China Grove Baptist Church
  [Mississippi          | Adams County Missionary Baptist Associat]  Natchez; Rev. C. R. Anderson
  [Mississippi          | Adams County Missionary Baptist Associat]  10 Keim Ave., N

### 2.4 Definitions of removed-line types

Each removed line is tagged with one of the following types:

| Type | What it is | Example |
|------|-----------|---------|
| `state_header` | Standalone state name | "Mississippi" |
| `assoc_declaration` | First line of an association/convention block | "Adams County Missionary Baptist Association" |
| `assoc_continuation` | Additional association name lines within a block | "General Missionary Baptist State Convention" |
| `officer_line` | Line containing Moderator/Secretary/President | "Rev. A. A. Cosey, President" |
| `officer_address` | Address line within an officer block | "107 E. Percy St., Greenwood" |
| `state_in_block` | State name embedded in an officer block | "Mississippi" |
| `assoc_running_header` | Association name repeated at page top (no officers) | "Adams County Missionary Baptist Association" |
| `assoc_fragment` | Partial association name (District/County/etc.) | "Bolivar County Missionary" |
| `stray_officer` | Officer line not part of a recognized block | "Rev. H. G. Gardner, Moderator" |

## Step 3: Group Lines into Church Entries

Each church entry spans 1–5 lines in the cleaned text (church name, location, pastor, pastor address). We detect church-name lines and group continuation lines with them.

### 3.1 Detect church-name lines

A line starts a new church entry if it contains:
- `Church` — since this is a Baptist church directory, any line with "Church" (and OCR variants) is almost certainly the start of a new entry
- `Chapel`, `Temple`, `Tabernacle`, `Mission` — other building types that appear in the directory
- Lines ending with `Baptist` (no "Church" keyword) — truncated names like "East Hope Baptist"

Exclusion: street addresses containing these words (e.g., "Church St.") are not entry starts.

In [10]:
def is_church_line(line):
    """Detect if a line is the start of a new church entry."""
    # Street-address exclusion pattern (reused below)
    street_pat = r"(St|Ave|Rd|Street|Blvd)"

    # "Church" (including OCR variant "Chur ch")
    if re.search(r"\bChur\s*ch\b", line) and not re.search(r"Chur\s*ch\s+" + street_pat, line):
        return True
    # OCR misspellings of "Church": Churoh, Chruch, @hurch, Churah, Churchi
    if re.search(r"\b(Churoh|Chruch|Churah|Churchi)\b|@hurch", line):
        return True
    # "Baptist Burch" — OCR for "Baptist Church"
    if re.search(r"Baptist\s+Burch\b", line):
        return True
    # "Chapel" and OCR variant "Chapol"
    if re.search(r"\b(Chapel|Chapol)\b", line) and not re.search(r"(Chapel|Chapol)\s+" + street_pat, line):
        return True
    # "Temple"
    if re.search(r"\bTemple\b", line) and not re.search(r"Temple\s+" + street_pat, line):
        return True
    # "Tabernacle"
    if re.search(r"\bTabernacle\b", line) and not re.search(r"Tabernacle\s+" + street_pat, line):
        return True
    # "Mission"
    if re.search(r"\bMission\b", line) and not re.search(r"Mission\s+" + street_pat, line):
        return True
    # Line ending with "Baptist" (no "Church") — e.g., "East Hope Baptist"
    if re.search(r"Baptist\s*$", line) \
            and not re.search(ASSOC_RE, line) \
            and not any(kw in line for kw in ASSOC_FRAGMENT_KEYWORDS):
        return True
    return False

### 3.1b Remove noise lines (short OCR artifacts)

Very short lines (fewer than 3 alphabetic characters) that survive Step 2 are OCR artifacts.

In [11]:
def is_noise_line(line):
    """Return True if a line is too short or non-alphabetic to be real content."""
    alpha_only = re.sub(r"[^a-zA-Z]", "", line)
    return len(alpha_only) < 3

before = len(entries)
entries = [e for e in entries if not is_noise_line(e["text"])]
noise_removed = before - len(entries)
print(f"Removed {noise_removed} noise lines (< 3 alphabetic chars)")
print(f"{len(entries):,} entry lines remain")

Removed 41 noise lines (< 3 alphabetic chars)
18,457 entry lines remain


### 3.2 Group lines into entries

Walk through the entry lines sequentially. Start a new group each time a church-name line is detected. Lines that don't start a new entry are appended to the current entry.

In [12]:
church_entries = []
current_entry = None
orphan_lines = []

for e in entries:
    line = e["text"]

    if is_church_line(line):
        if current_entry is not None:
            church_entries.append(current_entry)
        current_entry = {
            "lines": [line],
            "state": e["state"],
            "association": e["association"],
        }
    else:
        if current_entry is not None:
            current_entry["lines"].append(line)
        else:
            orphan_lines.append(line)

if current_entry is not None:
    church_entries.append(current_entry)

print(f"Grouped into {len(church_entries):,} church entries")
print(f"Orphan lines (before first church): {len(orphan_lines)}")
if orphan_lines:
    for o in orphan_lines[:10]:
        print(f"  \"{o}\"")

# How many lines does each entry have?
line_counts = Counter(len(e["lines"]) for e in church_entries)
print(f"\nLines per entry:")
for n, count in sorted(line_counts.items()):
    print(f"  {n} line(s): {count:>5} entries")

Grouped into 7,440 church entries
Orphan lines (before first church): 0

Lines per entry:
  1 line(s):   327 entries
  2 line(s):  3680 entries
  3 line(s):  3018 entries
  4 line(s):   378 entries
  5 line(s):    25 entries
  6 line(s):     6 entries
  7 line(s):     5 entries
  8 line(s):     1 entries


### 3.3 Preview grouped entries

Entries with 5+ lines may indicate grouping issues worth inspecting.

In [13]:
sep = "=" * 60
for target_len in [1]:
    batch = [e for e in church_entries if len(e["lines"]) == target_len]
    print(f"\n{sep}")
    print(f"Entries with {target_len} lines: {len(batch)}")
    print(sep)
    for e in batch[:30]:
        st = e["state"] or "???"
        print(f"  [{st}] ({len(e['lines'])} lines)")
        for ln in e["lines"]:
            print(f"    {ln}")
        print()


Entries with 1 lines: 327
  [Mississippi] (1 lines)
    Second Pleasant Valley Baptist

  [Mississippi] (1 lines)
    Union Grove Baptist Church, Duncan

  [Mississippi] (1 lines)
    St. Paul Baptist Church, Pattison

  [Mississippi] (1 lines)
    Calvory Papeist Church, Laurel

  [Mississippi] (1 lines)
    Mt. Horeb Baptist Church, Hollandale

  [Mississippi] (1 lines)
    Mt. Risin Baptist Church, Tchula

  [Mississippi] (1 lines)
    Mercy Seat Baptist Church

  [Mississippi] (1 lines)
    Church Hill; Rev. P. E. Frisby

  [Mississippi] (1 lines)
    Mt. Hope Baptist Church, Canton

  [Mississippi] (1 lines)
    Stokes Chapel, Stokes

  [Mississippi] (1 lines)
    Oakland Baptist Church, Bovina

  [Mississippi] (1 lines)
    Jones Chapel, Schlater

  [Mississippi] (1 lines)
    Mt. Eva Baptist Church, Darling

  [Mississippi] (1 lines)
    Spring Hill Baptist

  [Mississippi] (1 lines)
    St. Mary Baptist Church, Gunnison

  [Mississippi] (1 lines)
    Mount Olive Mission

  [Mi

### 3.4 Investigate 1-line entries

Most church entries have 2–4 lines. Entries with only 1 line may indicate grouping errors or legitimate entries where only the church name was recorded.

In [14]:
one_line_entries = [(i, e) for i, e in enumerate(church_entries) if len(e["lines"]) == 1]
print(f"Total 1-line entries: {len(one_line_entries)}")
print()
for idx, (i, e) in enumerate(one_line_entries[:30]):
    st = e["state"] or "???"
    print(f"  #{idx+1:>3}  [{st:20s}]  {e['lines'][0]}")

Total 1-line entries: 327

  #  1  [Mississippi         ]  Second Pleasant Valley Baptist
  #  2  [Mississippi         ]  Union Grove Baptist Church, Duncan
  #  3  [Mississippi         ]  St. Paul Baptist Church, Pattison
  #  4  [Mississippi         ]  Calvory Papeist Church, Laurel
  #  5  [Mississippi         ]  Mt. Horeb Baptist Church, Hollandale
  #  6  [Mississippi         ]  Mt. Risin Baptist Church, Tchula
  #  7  [Mississippi         ]  Mercy Seat Baptist Church
  #  8  [Mississippi         ]  Church Hill; Rev. P. E. Frisby
  #  9  [Mississippi         ]  Mt. Hope Baptist Church, Canton
  # 10  [Mississippi         ]  Stokes Chapel, Stokes
  # 11  [Mississippi         ]  Oakland Baptist Church, Bovina
  # 12  [Mississippi         ]  Jones Chapel, Schlater
  # 13  [Mississippi         ]  Mt. Eva Baptist Church, Darling
  # 14  [Mississippi         ]  Spring Hill Baptist
  # 15  [Mississippi         ]  St. Mary Baptist Church, Gunnison
  # 16  [Mississippi         ]  Mount Oli

### 3.5 Summary after Step 3

At this point we have the initial grouping. Next steps:
- Manually check and fix entries with 5+ lines (Section 3.6+)
- Investigate and fix 1-line entries (Section 3.7+)
- Then proceed to Step 4 (parsing into structured columns)

### 3.6 Fix 5+ line entries (verified against PDF scans)

All 37 entries with 5+ lines were cross-checked against the PDF scans.
26 entries had errors (leaked headers, officer blocks, two-column interleaving, OCR garbling).
11 entries are legitimate 5-line entries and are left unchanged.

A CSV log of all corrections is saved alongside this notebook.

In [15]:
# --- Helper functions for manual corrections ---

def find_entry(first_line, state=None, assoc_contains=None, line_contains=None):
    """Find index of an entry by its first line, optionally filtering by state/association."""
    matches = []
    for i, e in enumerate(church_entries):
        if e["lines"][0] == first_line:
            if state and e["state"] != state:
                continue
            if assoc_contains and assoc_contains not in (e["association"] or ""):
                continue
            if line_contains and not any(line_contains in l for l in e["lines"]):
                continue
            matches.append(i)
    if len(matches) == 1:
        return matches[0]
    elif len(matches) == 0:
        print(f"  WARNING: no match for '{first_line}'")
        return None
    else:
        print(f"  WARNING: {len(matches)} matches for '{first_line}' — add state/assoc/line filter")
        return None

def split_entry(idx, split_at):
    """Split entry at idx into two. Lines[:split_at] stay, lines[split_at:] become new entry."""
    e = church_entries[idx]
    entry2 = {"lines": e["lines"][split_at:], "state": e["state"], "association": e["association"]}
    e["lines"] = e["lines"][:split_at]
    church_entries.insert(idx + 1, entry2)
    print(f"  Split: '{e['lines'][0]}' ({split_at} lines) + '{entry2['lines'][0]}' ({len(entry2['lines'])} lines)")

def trim_entry(idx, keep):
    """Keep only the first `keep` lines of entry at idx."""
    dropped = church_entries[idx]["lines"][keep:]
    church_entries[idx]["lines"] = church_entries[idx]["lines"][:keep]
    print(f"  Trimmed: '{church_entries[idx]['lines'][0]}' — dropped {dropped}")

def remove_lines(idx, line_indices):
    """Remove specific lines (by 0-based index) from entry at idx."""
    e = church_entries[idx]
    dropped = [e["lines"][i] for i in line_indices]
    e["lines"] = [l for i, l in enumerate(e["lines"]) if i not in line_indices]
    print(f"  Removed lines from '{e['lines'][0]}': {dropped}")

def fix_false_split(idx):
    """The last line of entry[idx] is actually the start of the next entry's name.
    Remove it from entry[idx] and prepend it to entry[idx+1]'s first line."""
    dangling = church_entries[idx]["lines"].pop()
    church_entries[idx + 1]["lines"][0] = dangling + " " + church_entries[idx + 1]["lines"][0]
    print(f"  Fixed false split: '{church_entries[idx+1]['lines'][0]}'")


# =====================================================================
# Apply corrections (all verified against PDF scans)
# =====================================================================
before = len(church_entries)
print("Applying manual corrections for 5+ line entries...\n")

# --- 8-line: Tulane Baptist Church (MS, PDF p.38, #46) ---
# Lines 2-7 are leaked convention officer block (Prosident/Socretary OCR variants escaped Step 2)
idx = find_entry("Tulane Baptist Church", state="Mississippi", assoc_contains="Yazoo")
if idx is not None:
    remove_lines(idx, [2, 3, 4, 5, 6, 7])

# --- 7-line: Wilder Avenue Baptist Church (MT, PDF p.45, #53) ---
# Lines 3-6 are leaked NEBRASKA header + officer block (MEBRASKA/Loderator OCR variants escaped Step 2)
idx = find_entry("Wilder Avenue Baptist Church", state="Montana")
if idx is not None:
    trim_entry(idx, 3)

# --- 7-line: Eastern Chapel (NC, PDF p.89, #97) ---
# Lines 0-1 are Eastern Chapel. Lines 2-4 are garbled headers. Lines 5-6 are Edson Chapel.
idx = find_entry("Eastern Chapel; Rev. J. T. Deans", state="North Carolina")
if idx is not None:
    e = church_entries[idx]
    new_entry = {"lines": ["Edson Chapel, Warsaw", "Rev. J. M. Newkirk, Rosehill"],
                 "state": e["state"], "association": e["association"]}
    e["lines"] = e["lines"][:2]
    church_entries.insert(idx + 1, new_entry)
    print(f"  Trimmed Eastern Chapel to 2 lines, created Edson Chapel (2 lines)")

# --- 7-line: Calvary Baptist Church (OH, PDF p.112, #120) ---
# Lines 1-3 are leaked officer address lines from Northwestern District Baptist Association
idx = find_entry("Calvary Baptist Church", state="Ohio", assoc_contains="Northwestern")
if idx is not None:
    remove_lines(idx, [1, 2, 3])

# --- 7-line: First Baptist Church, 27th St. (DC/WA, PDF p.288, #296) ---
# Lines 4-6 are from a missing Carron Baptist Church entry (two-column interleave)
idx = find_entry("First Baptist Church, 27th St. &", state="Washington", assoc_contains="General Baptist")
if idx is not None:
    trim_entry(idx, 4)

# --- 7-line: Paramount Baptist Church (D.C., PDF p.291, #299) ---
# Lines 4-6 are section header remnants (Washington D.C. + Unaffiliated Churches x2)
idx = find_entry("Paramount Baptist Church", state="D. C.")
if idx is not None:
    trim_entry(idx, 4)

# --- 6-line: Friendship Baptist Church, 131st (NY, PDF p.71, #79) ---
# Lines 4-5 are garbled running header ("Manhattan, Westchester..." association name)
idx = find_entry("Friendship Baptist Church", state="New York", assoc_contains="Manhattan")
if idx is not None:
    trim_entry(idx, 4)

# --- 6-line: Lees Cross Roads Baptist Church (NC, PDF p.98, #106) ---
# Lines 3-5 are Mt. Carmel Pine Level entry (missing "Church" keyword, not caught)
idx = find_entry("Lees Cross Roads Baptist Church", state="North Carolina")
if idx is not None:
    split_entry(idx, 3)

# --- 6-line: Antioch Baptist Church (OK, PDF p.123, #131) ---
# Lines 1-2 are leaked secretary address from East Zion District Association
idx = find_entry("Antioch Baptist Church", state="Oklahoma", assoc_contains="East Zion")
if idx is not None:
    remove_lines(idx, [1, 2])

# --- 6-line: Tucker Baptist Church (TN, PDF p.164, #172) ---
# Lines 3-5 are leaked officer block (Moderstor OCR variant escaped Step 2)
idx = find_entry("Tucker Baptist Church", state="Tennessee")
if idx is not None:
    trim_entry(idx, 3)

# --- 6-line: Delaware Avenue Baptist Church (DC/WA, PDF p.288, #296) ---
# Lines 4-5 are from a missing First New Hope Baptist entry (two-column interleave)
idx = find_entry("Delaware Avenue Baptist Church", state="Washington")
if idx is not None:
    trim_entry(idx, 4)

# --- 6-line: New Bethel Baptist Church (D.C., PDF p.291, #299) ---
# Lines 4-5 are section header + running header (Unaffiliated Churches / Washington, D.C.)
idx = find_entry("New Bethel Baptist Church", state="D. C.", line_contains="812 S St.")
if idx is not None:
    trim_entry(idx, 4)

# --- 5-line: Shiloh Baptist Church, Rockville Center (NY, PDF p.69, #77) ---
# Line 4 is running header fragment from association name
idx = find_entry("Shiloh Baptist Church", state="New York", assoc_contains="Eastern", line_contains="Rockville")
if idx is not None:
    trim_entry(idx, 4)

# --- 5-line: Ebenezer Baptist Church, Poughkeepsie (NY, PDF p.75, #83) ---
# Line 4 is leaked pastor line from Fairmont Baptist Church
idx = find_entry("Ebenezer Baptist Church", state="New York", line_contains="Poughkeepsie")
if idx is not None:
    trim_entry(idx, 4)

# --- 5-line: Friendship Betist Church (NC, PDF p.89, #97) ---
# Severely garbled OCR. Lines 0-2 = Friendship Baptist Church (fix OCR). Lines 3-4 = Halle Chapel.
idx = find_entry("Friendship Betist Church", state="North Carolina")
if idx is not None:
    e = church_entries[idx]
    e["lines"][0] = "Friendship Baptist Church"
    e["lines"][1] = "Rocky Point; Rev. Joseph Miller"
    e["lines"][2] = "R. F. D. 1, Wilmington"
    new_entry = {"lines": ["Halle Chapel; Rev. S. V. Wells", "Box 692, Warsaw"],
                 "state": e["state"], "association": e["association"]}
    e["lines"] = e["lines"][:3]
    church_entries.insert(idx + 1, new_entry)
    print(f"  Fixed OCR in Friendship Baptist, created Halle Chapel (2 lines)")

# --- 5-line: Belay 110 Fantist Church (NC, PDF p.89, #97) ---
# Severely garbled OCR. Lines 0-2 = Beulaville Baptist Church (fix OCR). Lines 3-4 = Hill's Chapel.
idx = find_entry("Belay 110 Fantist Church", state="North Carolina")
if idx is not None:
    e = church_entries[idx]
    e["lines"][0] = "Beulaville Baptist Church"
    e["lines"][1] = "Beulaville; Rev. J. W. Dell"
    e["lines"][2] = "Willard"
    new_entry = {"lines": ["Hill's Chapel, Pender", "Rev. S. M. White, Wilmington"],
                 "state": e["state"], "association": e["association"]}
    e["lines"] = e["lines"][:3]
    church_entries.insert(idx + 1, new_entry)
    print(f"  Fixed OCR in Beulaville Baptist, created Hill's Chapel (2 lines)")

# --- 5-line: Walnut Grove Raptist Church (NC, PDF p.94, #102) ---
# Lines 3-4 are leaked officer block from Rowan Baptist Association
idx = find_entry("Walnut Grove Raptist Church", state="North Carolina")
if idx is not None:
    trim_entry(idx, 3)

# --- 5-line: Mt. Hebron Baptist Church, Ellorco (SC, PDF p.150, #158) ---
# Lines 2-4 are Mt. Salem Baptist Church entry (truncated "Chur" missed by is_church_line)
idx = find_entry("Mt. Hebron Baptist Church, Ellorco", state="South Carolina")
if idx is not None:
    e = church_entries[idx]
    e["lines"][0] = "Mt. Hebron Baptist Church, Elloree"
    e["lines"][1] = "Rev. J. M. Felder, Elloree"
    new_lines = ["Mt. Salem Baptist Church",
                 "St. Matthews; Rev. S. J. Jackson",
                 "Fort Motte"]
    new_entry = {"lines": new_lines, "state": e["state"], "association": e["association"]}
    e["lines"] = e["lines"][:2]
    church_entries.insert(idx + 1, new_entry)
    print(f"  Fixed OCR in Mt. Hebron, created Mt. Salem Baptist Church (3 lines)")

# --- 5-line: Mt. Carmel Baptist Church, 16th (TN, PDF p.164, #172) ---
# Line 4 is association name fragment
idx = find_entry("Mt. Carmel Baptist Church", state="Tennessee", line_contains="16th")
if idx is not None:
    trim_entry(idx, 4)

# --- 5-line: Sixth Mount Zion Baptist Church (VA, PDF p.244, #252) ---
# Line 4 is "Independent Churches" section header
idx = find_entry("Sixth Mount Zion Baptist Church", state="Virginia")
if idx is not None:
    trim_entry(idx, 4)

# --- 5-line: Second Baptist Church, Everett (WA, PDF p.283, #291) ---
# Lines 3-4 are leaked DC section headers
idx = find_entry("Second Baptist Church", state="Washington", line_contains="Everett")
if idx is not None:
    trim_entry(idx, 3)

# --- 5-line: Northeastern Baptist Church (D.C., PDF p.291, #299) ---
# Line 4 is start of "North West Beulah Independent Baptist Church" name split across two lines
idx = find_entry("Northeastern Baptist Church", state="D. C.")
if idx is not None:
    fix_false_split(idx)

# --- 5-line: Springfield Baptist Church (WV, PDF p.302, #310) ---
# Lines 3-4 are Striving Valley church (missing "Church" keyword, not caught)
idx = find_entry("Springfield Baptist Church", state="West Virginia", line_contains="Springton")
if idx is not None:
    split_entry(idx, 3)

# --- 5-line: Welcome Baptist Church, Beckley (WV, PDF p.307, #315) ---
# Lines 2-4 are APPENDIX header + association name fragments
idx = find_entry("Welcome Baptist Church, Beckley", state="West Virginia")
if idx is not None:
    trim_entry(idx, 2)

# --- 5-line: New Hope Baptist Church, Braddock (PA, PDF p.310, #318) ---
# Lines 3-4 are association name fragments ("Western Pennsylvania and / Eastern Ohio")
idx = find_entry("New Hope Baptist Church, Braddock", state="Washington",
                 assoc_contains="Baptist Association of Western")
if idx is not None:
    trim_entry(idx, 3)

# --- 5-line: Shiloh Baptist Church, Pittsburgh (PA, PDF p.313, #323) ---
# Line 4 is association name fragment ("Youghiegeny Western")
idx = find_entry("Shiloh Baptist Church, Pittsburgh", state="Pennsylvania",
                 assoc_contains="Union Baptist")
if idx is not None:
    trim_entry(idx, 4)


# =====================================================================
# Summary
# =====================================================================
after = len(church_entries)
print(f"\nDone. Entries before: {before}, after: {after} (net change: {after - before:+d})")

line_counts = Counter(len(e["lines"]) for e in church_entries)
print(f"\nUpdated lines per entry:")
for n, count in sorted(line_counts.items()):
    print(f"  {n} line(s): {count:>5} entries")

# Verify no 6+ line entries remain
long_entries = [(i, e) for i, e in enumerate(church_entries) if len(e["lines"]) >= 6]
if long_entries:
    print(f"\nWARNING: {len(long_entries)} entries still have 6+ lines!")
    for i, e in long_entries:
        print(f"  #{i} [{e['state']}] {len(e['lines'])} lines: {e['lines'][0]}")
else:
    print("\nAll entries now have 5 or fewer lines. ✓")


Applying manual corrections for 5+ line entries...

  Removed lines from 'Tulane Baptist Church': ['Rev. Benjamin J. Perkins, Prosident', 'Cleveland, Ohio', 'Winona', 'Rov. R. Rutherford, Recording Socretary', 'Grace', 'Winterville']
  Trimmed: 'Wilder Avenue Baptist Church' — dropped ['MEBRASKA', 'Rev. F. P. Jones, Loderator', '2422 Ohio St., Omaha', '2407 . 22nd St., Omaha']
  Trimmed Eastern Chapel to 2 lines, created Edson Chapel (2 lines)
  Removed lines from 'Calvary Baptist Church': ['704 Collingwood Blvd.', 'Toledo', '721 Vance St., Toledo']
  Trimmed: 'First Baptist Church, 27th St. &' — dropped ['935 1/2 Florida Ave. N. W.', 'Rev. Patrick Yancy', '1914 New Hampshire Ave.']
  Trimmed: 'Paramount Baptist Church' — dropped ['Washington D. C.', 'Una ffiliated Churches', 'Unaffiliated Churches']
  Trimmed: 'Friendship Baptist Church' — dropped ['Farhatten, Westchester, Bronx and', 'Sabor island Lecuciation']
  Split: 'Lees Cross Roads Baptist Church' (3 lines) + 'Mt. Carmel, Pine 

### 3.7 Fix 1-line entries: Baptist$ association fragments and split names

Following Volume 1's pattern, the first pass over the 1-line entries handles those whose
text ends in `Baptist` with no `Church` keyword:

- **Pattern A — association fragment (remove):** an association header such as
  `"X Baptist Association"` whose `"Association"` line was stripped in Step 2, leaving the
  `"X Baptist"` fragment grouped onto the following church.
- **Pattern B — split church name (merge):** OCR broke a church name across two lines, so the
  *next* entry begins with `"Church..."`. We splice the fragment back onto it.

The rule (verified against the PDF scans / raw OCR): if the next entry starts with `Church`
it is a split name to merge; otherwise it is an association fragment to drop.

In [16]:
# ===================================================================
# Section 3.7 -- Fix 1-line entries ending in "Baptist" (no "Church")
#   Pattern B: the NEXT entry starts with "Church" -> OCR split the church
#              name across two lines; merge the fragment back on.
#   Pattern A: otherwise -> association-header fragment ("X Baptist
#              Association" with "Association" stripped by Step 2); remove.
# Mechanical pass over ALL such 1-line entries in the volume.
# ===================================================================
before = len(church_entries)
baptist_one_liners = [i for i, e in enumerate(church_entries)
                      if len(e["lines"]) == 1
                      and re.search(r"Baptist\s*$", e["lines"][0])
                      and not re.search(r"\bChur\s*ch\b", e["lines"][0])]

split_fixes, to_remove = [], []
for idx in baptist_one_liners:
    nxt = idx + 1
    if nxt < len(church_entries) and re.match(r"^Church\b", church_entries[nxt]["lines"][0]):
        split_fixes.append(idx)        # Pattern B
    else:
        to_remove.append(idx)          # Pattern A

print(f"Baptist$ 1-line entries: {len(baptist_one_liners)}  "
      f"(merge {len(split_fixes)}, remove {len(to_remove)})")

# Pattern B: prepend the name fragment onto the continuation entry
for idx in split_fixes:
    church_entries[idx + 1]["lines"][0] = (
        church_entries[idx]["lines"][0] + " " + church_entries[idx + 1]["lines"][0])
    print(f"  merge -> {church_entries[idx + 1]['lines'][0]!r}")

# Remove association fragments + now-redundant split stubs (descending order)
to_remove.extend(split_fixes)
for idx in sorted(set(to_remove), reverse=True):
    church_entries.pop(idx)

print(f"\nEntries: {before} -> {len(church_entries)}")
print("Remaining 1-line entries:",
      sum(1 for e in church_entries if len(e["lines"]) == 1))

Baptist$ 1-line entries: 67  (merge 11, remove 56)
  merge -> 'Second Pleasant Valley Baptist Church, Duncan; Rev. D. L. Clifton'
  merge -> 'Thompson Memorial Community Baptist Church, Auburn; Rev. William T. Rives'
  merge -> 'Pleasant Hill Robeson Baptist Church, Lumberton'
  merge -> 'Mt. Hope Springhill Baptist Church, Columbia; Rev. C. H. Smith'
  merge -> 'Second Pleasant Grove Baptist Church, Houston; Rov. F. Turner'
  merge -> 'Second Mount Calvary Baptist Church, Milford'
  merge -> 'Bethel Institutional Baptist Church, Charlottesville'
  merge -> 'First Rising Mount Zion Baptist Church, 1609 11th St. N. W.'
  merge -> 'First Marshall Heights Baptist Church, 50th & B Sts., S. E.'
  merge -> 'Good Samaritan Community Baptist Church, 413 Franklin St., N. W.'
  merge -> 'New Saint John Independent Baptist Church, 5204 Foote St. N. E.'

Entries: 7446 -> 7379
Remaining 1-line entries: 260


### 3.8 Fix remaining 1-line entries #1–100 (verified against PDF scans)

Each of the first 100 remaining 1-line entries was cross-checked against the HathiTrust PDF
scan (`mdp.39015024219845`, PDF page = printed page + 9). The large majority are
**genuinely short** (church name + town only, no pastor printed) and need no change. The
corrections below cover the entries where OCR lost or scrambled data:

- **APPEND** — pastor/location/address line dropped by Step 2 (officer-like line, county
  line, or swallowed by an association officer block).
- **RENAME** — OCR typo in an otherwise-complete short entry.
- **COLFIX** — two-column OCR scramble: a location/pastor line was mis-detected as a church
  (false split) or interleaved into a neighbouring entry. These also repair the neighbour.
- **REMOVE** — association fragment not caught by the `Baptist$` rule.

Entries are located by text (+ state/association) so the edits are robust to index shifts.

In [17]:
# ===================================================================
# Section 3.8 -- Manual PDF-verified corrections for 1-line entries #1-100
#   (reuses find_entry() from Section 3.6)
# ===================================================================
def find_oneliner(first_line, state=None, assoc_contains=None):
    m = [i for i, e in enumerate(church_entries)
         if len(e["lines"]) == 1 and e["lines"][0] == first_line
         and (state is None or e["state"] == state)
         and (assoc_contains is None or assoc_contains in (e["association"] or ""))]
    if len(m) != 1:
        print(f"  !! {len(m)} matches for 1-liner {first_line!r}")
    return m[0] if m else None

def do_append(first_line, new_lines, state=None, assoc=None):
    i = find_oneliner(first_line, state, assoc)
    if i is not None:
        church_entries[i]["lines"].extend(new_lines)
        print(f"  APPEND {first_line!r} <- {new_lines}")

def do_rename(first_line, new_name, state=None, assoc=None):
    i = find_oneliner(first_line, state, assoc)
    if i is not None:
        church_entries[i]["lines"][0] = new_name
        print(f"  RENAME {first_line!r} -> {new_name!r}")

def fix_neighbor_line(first_line, old_sub, new_val, state=None, assoc=None, line_contains=None):
    i = find_entry(first_line, state, assoc, line_contains)
    if i is None:
        return
    for k, l in enumerate(church_entries[i]["lines"]):
        if old_sub in l:
            church_entries[i]["lines"][k] = new_val
            print(f"  NBR-FIX {first_line!r} line[{k}] -> {new_val!r}")
            return
    print(f"  !! neighbor line {old_sub!r} not found in {first_line!r}")

def remove_neighbor_line(first_line, sub, state=None, assoc=None, line_contains=None):
    i = find_entry(first_line, state, assoc, line_contains)
    if i is None:
        return
    for k, l in enumerate(church_entries[i]["lines"]):
        if sub in l:
            print(f"  NBR-DROP {first_line!r} dropped {church_entries[i]['lines'].pop(k)!r}")
            return
    print(f"  !! neighbor drop {sub!r} not found in {first_line!r}")

before = len(church_entries)

# --- APPEND: pastor/location/address dropped by Step 2 ---
do_append("Union Grove Baptist Church, Duncan", ["Rev. W. M. President"],
          "Mississippi", "Bolivar County Missionary")                                   # #2
do_append("St. Peter Baptist Church", ["Oconee County"],
          "South Carolina", "Seneca River")                                             # #96
do_append("Adams Chapel", ["Springfield; Rev. R. L. West", "1500 Cheatman St., Springfield"],
          "Tennessee", "Brown's Creek")                                                 # #100

# --- RENAME: OCR typo (entry otherwise genuinely short) ---
do_rename("Calvory Papeist Church, Laurel", "Calvary Baptist Church, Laurel",
          "Mississippi", "First Enterprise")                                            # #4
do_rename('Pilgrim Baptist Church, Cincinnati "', "Pilgrim Baptist Church, Cincinnati",
          "Ohio", "Western Union")                                                      # #63
do_rename("Third Zion Baptist Church, Xonia", "Third Zion Baptist Church, Xenia",
          "Ohio", "Western Union")                                                      # #65

# --- COLFIX: false split (a location line mis-detected as a church) ---
# #7/#8: "Church Hill; Rev. P. E. Frisby" is Mercy Seat's location/pastor line
i_ms = find_oneliner("Mercy Seat Baptist Church", "Mississippi", "Jefferson County")
if i_ms is not None:
    church_entries[i_ms]["lines"].append("Church Hill; Rev. P. E. Frisby")
    print("  COLFIX Mercy Seat Baptist Church <- 'Church Hill; Rev. P. E. Frisby'")
i_ch = find_oneliner("Church Hill; Rev. P. E. Frisby", "Mississippi", "Jefferson County")
if i_ch is not None:
    church_entries.pop(i_ch)
    print("  REMOVE stub 'Church Hill; Rev. P. E. Frisby' (merged into Mercy Seat)")

# #23/#24: NY column scramble (Fairmont Baptist <-> Bethany Baptist, New Rochelle)
do_append("Fairmont Baptist Church",
          ["17 Rockland St., Haverstraw", "Rev. Robert Harrell, Haverstraw"],
          "New York", "Western New York")                                               # #23
do_rename("Bethany Baptist Church, New Rochelle 17 Rockland St., Haverst raw",
          "Bethany Baptist Church, New Rochelle", "New York", "Western New York")        # #24

# #28: NC row-interleave (Brister Creek / Mt. Zion, Chadbourn)
do_append("Brister Creek Baptist Church", ["Tabor City; Rev. S. J. Bryant, Supply"],
          "North Carolina", "Waccamsaw")
fix_neighbor_line("Mt. Zion Baptist Church, Chadbourn",
                  "Tabor City; Rev. S. J. Bryant, Supply Rev. W. M. Falk",
                  "Rev. W. M. Falk, Chadbourn", "North Carolina", "Waccamsaw",
                  line_contains="Tabor City; Rev. S. J. Bryant, Supply Rev. W. M. Falk")

# #29: NC row-interleave (Diamond Branch / St. Bethel, Whiteville)
do_append("Diamond Branch Baptist Church", ["Whiteville; Rev. A. T. Graham, Conway"],
          "North Carolina", "Waccamsaw")
fix_neighbor_line("St. Bethel Baptist Church, Whiteville",
                  "Whiteville; Rev. A. T. Graham, Conway Rev. W. M. Jones",
                  "Rev. W. M. Jones, Hallsboro", "North Carolina", "Waccamsaw",
                  line_contains="Whiteville; Rev. A. T. Graham, Conway Rev. W. M. Jones")

# #36: NC row-interleave (Barringer's Chapel / Elizabeth, Pee Dee)
do_append("Barringer's Chapel, Norwood", ["Rev. J. V. Esterling, Kollock, S. C."],
          "North Carolina", "Zion Missionary")
fix_neighbor_line("Elizabeth Baptist Church, Pee Dee",
                  "Rev. J. V. Esterling, Kollock, S. C. Rev. J. A. Ingram",
                  "Rev. J. A. Ingram, Lilesville", "North Carolina", "Zion Missionary",
                  line_contains="Rev. J. V. Esterling, Kollock, S. C. Rev. J. A. Ingram")

# #66: OK row-interleave (Gilfield, Davis / Bethlehem, Pauls Valley) + name OCR typo
do_rename("Gilfield Baprist Church, Davis", "Gilfield Baptist Church, Davis",
          "Oklahoma", "Chickasaw")
i_g = find_entry("Gilfield Baptist Church, Davis", "Oklahoma", "Chickasaw")
if i_g is not None:
    church_entries[i_g]["lines"].append("Rev. D. B. Hill")
    print("  APPEND Gilfield Baptist Church, Davis <- 'Rev. D. B. Hill'")
fix_neighbor_line("Bethlehem Baptist Church, Pauls Valle Rev. D. B. Hill",
                  "Bethlehem Baptist Church, Pauls Valle",
                  "Bethlehem Baptist Church, Pauls Valley", "Oklahoma", "Chickasaw")

# #72: OK 3-way scramble (Sunrise, Vinita / St. Luke, Canadian / St. Paul, Webbers)
do_append("Sunrise Baptist Church, Vinita", ["Rev. W. M. Harris", "1019 S. 12th St., Muskogee"],
          "Oklahoma", "Collate")
fix_neighbor_line("St. Luke Baptist Church, Canadian",
                  "Rev. C. R. Mason, Gen. Del. McAlester Rev. W. M. Harris",
                  "Rev. C. R. Mason, Gen. Del. McAlester", "Oklahoma", "Collate",
                  line_contains="Rev. C. R. Mason, Gen. Del. McAlester Rev. W. M. Harris")
remove_neighbor_line("St. Paul Baptist Church, Webbers", "1019 S. 12th St., Muskogee",
                     "Oklahoma", "Collate", line_contains="1019 S. 12th St., Muskogee")

# --- REMOVE: association fragment not ending in "Baptist" ---
i_rm = find_oneliner("Mount Olive Mission", "Mississippi", "Gethsemane Mount Moriah")   # #16
if i_rm is not None:
    church_entries.pop(i_rm)
    print("  REMOVE 'Mount Olive Mission' (assoc fragment)")

# ===================================================================
# Section 3.8 (batch 2) -- 1-line entries #101-133, PDF-verified
#   (the rest are genuinely short or handled by 3.7's Baptist$ rule)
# ===================================================================
# COLFIX false-split: a location/pastor/address line mis-read as a church
# #108: "Morris Chapel; Rev. J. M. Williams" is New Zeal's location/pastor line
i = find_oneliner("New Zeal Baptist Church", "Tennessee", "Tennessee River District")
if i is not None:
    church_entries[i]["lines"].extend(["Morris Chapel; Rev. J. M. Williams", "Jackson"])
    print("  COLFIX New Zeal Baptist Church <- ['Morris Chapel; Rev. J. M. Williams', 'Jackson']")
j = find_entry("Morris Chapel; Rev. J. M. Williams", "Tennessee", "Tennessee River District")
if j is not None:
    church_entries.pop(j)
    print("  REMOVE stub 'Morris Chapel; Rev. J. M. Williams' (merged into New Zeal)")

# #119/#120: "Rev. J. H. Chapel, Seguin" is Zion Hill's pastor ('Chapel' -> false church)
i = find_oneliner("Zion Hill Baptist Church, Seguin", "Texas", "Regular Guadalupe")
if i is not None:
    church_entries[i]["lines"].append("Rev. J. H. Chapel, Seguin")
    print("  COLFIX Zion Hill Baptist Church, Seguin <- 'Rev. J. H. Chapel, Seguin'")
j = find_oneliner("Rev. J. H. Chapel, Seguin", "Texas", "Regular Guadalupe")
if j is not None:
    church_entries.pop(j)
    print("  REMOVE stub 'Rev. J. H. Chapel, Seguin' (merged into Zion Hill)")

# #127: "1009 S. 12th St., Temple" orphan address (is_church_line matched 'St.')
i = find_entry("Magnolia Baptist Church", "Texas", "La Grange",
               line_contains="Belton; Rev. C. S. Williamson")
if i is not None:
    church_entries[i]["lines"].append("1009 S. 12th St., Temple")
    print("  COLFIX Magnolia Baptist Church <- '1009 S. 12th St., Temple'")
j = find_oneliner("1009 S. 12th St., Temple", "Texas", "La Grange")
if j is not None:
    church_entries.pop(j)
    print("  REMOVE orphan '1009 S. 12th St., Temple' (merged into Magnolia)")

# RENAME (OCR typo; entry otherwise genuinely short)
do_rename("Smith Chapol, Columbus", "Smith Chapel, Columbus", "Texas", "La Grange")     # #132

print(f"\nEntries: {before} -> {len(church_entries)}")
print("Remaining 1-line entries:",
      sum(1 for e in church_entries if len(e["lines"]) == 1))
print("Lines per entry:",
      dict(sorted(Counter(len(e["lines"]) for e in church_entries).items())))

  APPEND 'Union Grove Baptist Church, Duncan' <- ['Rev. W. M. President']
  APPEND 'St. Peter Baptist Church' <- ['Oconee County']
  APPEND 'Adams Chapel' <- ['Springfield; Rev. R. L. West', '1500 Cheatman St., Springfield']
  RENAME 'Calvory Papeist Church, Laurel' -> 'Calvary Baptist Church, Laurel'
  RENAME 'Pilgrim Baptist Church, Cincinnati "' -> 'Pilgrim Baptist Church, Cincinnati'
  RENAME 'Third Zion Baptist Church, Xonia' -> 'Third Zion Baptist Church, Xenia'
  COLFIX Mercy Seat Baptist Church <- 'Church Hill; Rev. P. E. Frisby'
  REMOVE stub 'Church Hill; Rev. P. E. Frisby' (merged into Mercy Seat)
  APPEND 'Fairmont Baptist Church' <- ['17 Rockland St., Haverstraw', 'Rev. Robert Harrell, Haverstraw']
  RENAME 'Bethany Baptist Church, New Rochelle 17 Rockland St., Haverst raw' -> 'Bethany Baptist Church, New Rochelle'
  APPEND 'Brister Creek Baptist Church' <- ['Tabor City; Rev. S. J. Bryant, Supply']
  NBR-FIX 'Mt. Zion Baptist Church, Chadbourn' line[1] -> 'Rev. W. M. Falk,

### 3.9 Fix remaining 1-line entries #134–244 (verified against PDF scans)

**Fixes identified (42 total):**

- **COLFIX via "Temple"** (7 fixes): TX town name "Temple" triggers `is_church_line`,
  splitting pastor/address lines off their churches. Entries: Maud Chapel, Pine Bluff Baptist,
  Eighth Street Baptist, Heidenheimer Baptist (addr), Antioch Baptist (addr), Mt. Ozia Baptist,
  Mt. Zion Baptist (3-way).
- **COLFIX via "Chapel"** (1 fix): "Chapel Hill" location for Ebenezer Baptist Church.
- **COLFIX via "Church" in VA place names** (9 fixes): "Falls Church", "Church Roads",
  "Christ Church", "Church Road" are VA communities; "Church" triggers `is_church_line`.
  Entries: Chantilly Baptist, Oak Shade Baptist, Second Baptist (Falls Church), Shiloh Baptist
  (Ordricks), Springfield Baptist, Union Gospel Baptist, Calvary Baptist (Southside Rapp.),
  Grafton Baptist, Marmora Baptist (3-way).
- **RENAME** (4 fixes): OCR typos — "Forrest on"→"Forreston", "Falls Chur ch"→"Falls Church",
  "New Elan…I anexa"→"New Elam…Ianexa", "Feach Grove"→"Peach Grove".
- **APPEND county lines** (17 fixes): Star of the East Baptist Association lists churches
  with county location lines; all stripped by Step 2.
- **APPEND missing lines** (4 fixes): Southside Rappahannock (Grafton + Mt. Olive missing
  "Middlesex County"); D.C. (Carron Baptist + First New Hope Baptist — pastor/address
  lines removed by Step 2 from two-column OCR interleave).

In [18]:
# ===================================================================
# Section 3.9 -- Manual PDF-verified corrections for 1-line entries #134-244
# ===================================================================
before = len(church_entries)

# ------------------------------------------------------------------
# A. COLFIX: false splits caused by "Temple" (TX town name)
# ------------------------------------------------------------------
# #95: Maud Chapel, Maud <- "Rev. Tom Temple, Redwater"
do_append("Maud Chapel, Maud", ["Rev. Tom Temple, Redwater"],
          "Texas", "Unity")
i = find_oneliner("Rev. Tom Temple, Redwater", "Texas", "Unity")
if i is not None:
    church_entries.pop(i); print("  REMOVE stub 'Rev. Tom Temple, Redwater'")

# #98: Pine Bluff Baptist Church <- merge next entry "Hughes Springs; Rev. Thomas Temple / Redwater"
i = find_oneliner("Pine Bluff Baptist Church", "Texas", "Unity")
if i is not None:
    j = find_entry("Hughes Springs; Rev. Thomas Temple", "Texas", "Unity")
    if j is not None:
        church_entries[i]["lines"].extend(church_entries[j]["lines"])
        print(f"  COLFIX Pine Bluff <- {church_entries[j]['lines']}")
        church_entries.pop(j)
        print("  REMOVE stub 'Hughes Springs; Rev. Thomas Temple'")

# #105: Eighth Street Baptist Church <- merge "Temple; Rev. D. P. Prownell / 203 S. 10th Ave., Dallas"
i = find_oneliner("Eighth Street Baptist Church", "Texas", "Galilee-Griggs")
if i is not None:
    j = find_entry("Temple; Rev. D. P. Prownell", "Texas", "Galilee-Griggs")
    if j is not None:
        church_entries[i]["lines"].extend(church_entries[j]["lines"])
        print(f"  COLFIX Eighth Street <- {church_entries[j]['lines']}")
        church_entries.pop(j)
        print("  REMOVE stub 'Temple; Rev. D. P. Prownell'")

# #122: R.F.D. 3, Temple -> append to Heidenheimer Baptist Church
i = find_entry("Heidenheimer Baptist Church", "Texas", "St. Emanuel",
               line_contains="Heidenheimer; Rev. M. C. Moore")
if i is not None:
    church_entries[i]["lines"].append("R.F.D. 3, Temple")
    print("  COLFIX Heidenheimer <- 'R.F.D. 3, Temple'")
j = find_oneliner("R.F.D. 3, Temple", "Texas", "St. Emanuel")
if j is not None:
    church_entries.pop(j); print("  REMOVE orphan 'R.F.D. 3, Temple'")

# #124: Temple -> append to Antioch Baptist Church (Heidenheimer)
i = find_entry("Antioch Baptist Church", "Texas", "St. John",
               line_contains="Heidenheimer; Rev. L. Williams")
if i is not None:
    church_entries[i]["lines"].append("Temple")
    print("  COLFIX Antioch Baptist (Heidenheimer) <- 'Temple'")
j = find_oneliner("Temple", "Texas", "St. John")
if j is not None:
    church_entries.pop(j); print("  REMOVE orphan 'Temple'")

# #125/126: Rev. J. W. Wesley, Temple -> append to Mt. Ozia Baptist Church
do_append("Mt. Ozia Baptist Church, Rogers", ["Rev. J. W. Wesley, Temple"],
          "Texas", "St. John")
i = find_oneliner("Rev. J. W. Wesley, Temple", "Texas", "St. John")
if i is not None:
    church_entries.pop(i); print("  REMOVE stub 'Rev. J. W. Wesley, Temple'")

# #127/128/129: Mt. Zion Baptist Church + Temple; Rev. A. G. Harris + Temple (3-way)
i = find_oneliner("Mt. Zion Baptist Church", "Texas", "St. John")
if i is not None:
    church_entries[i]["lines"].extend(["Temple; Rev. A. G. Harris", "Temple"])
    print("  COLFIX Mt. Zion (St. John) <- ['Temple; Rev. A. G. Harris', 'Temple']")
# Remove the two stubs
for stub in ["Temple; Rev. A. G. Harris", "Temple"]:
    j = find_oneliner(stub, "Texas", "St. John")
    if j is not None:
        church_entries.pop(j); print(f"  REMOVE stub {stub!r}")

# ------------------------------------------------------------------
# B. COLFIX: false split caused by "Chapel" keyword
# ------------------------------------------------------------------
# #119/120: Ebenezer Baptist Church <- "Chapel Hill: Rev. C. C. Reed"
do_append("Ebenezer Baptist Church", ["Chapel Hill: Rev. C. C. Reed"],
          "Texas", "Lincoln Missionary")
i = find_oneliner("Chapel Hill: Rev. C. C. Reed", "Texas", "Lincoln Missionary")
if i is not None:
    church_entries.pop(i); print("  REMOVE stub 'Chapel Hill: Rev. C. C. Reed'")

# ------------------------------------------------------------------
# C. COLFIX: false splits caused by "Church" in VA place names
# ------------------------------------------------------------------
# #138/139: Chantilly Baptist Church <- "Rev. Oliver Hall, Falls Church"
do_append("Chantilly Baptist Church, Chantilly",
          ["Rev. Oliver Hall, Falls Church"], "Virginia", "Northern Virginia")
i = find_oneliner("Rev. Oliver Hall, Falls Church", "Virginia", "Northern Virginia")
if i is not None:
    church_entries.pop(i); print("  REMOVE stub 'Rev. Oliver Hall, Falls Church'")

# #142: "West Falls Church" -> append to Oak Shade Baptist Church
i = find_entry("Oak Shade Baptist Church", "Virginia", "Northern Virginia",
               line_contains="Catlett; Rev. S. W. Phillips")
if i is not None:
    church_entries[i]["lines"].append("West Falls Church")
    print("  COLFIX Oak Shade Baptist <- 'West Falls Church'")
j = find_oneliner("West Falls Church", "Virginia", "Northern Virginia")
if j is not None:
    church_entries.pop(j); print("  REMOVE stub 'West Falls Church'")

# #143/144: Second Baptist Church, Falls Church + pastor
do_rename("Second Baptist Church, Falls Chur ch",
          "Second Baptist Church, Falls Church", "Virginia", "Northern Virginia")
i = find_entry("Second Baptist Church, Falls Church", "Virginia", "Northern Virginia")
if i is not None:
    church_entries[i]["lines"].append("Rev. W. E. Costner, Falls Church")
    print("  COLFIX Second Baptist (Falls Church) <- pastor")
j = find_oneliner("Rev. W. E. Costner, Falls Church", "Virginia", "Northern Virginia")
if j is not None:
    church_entries.pop(j); print("  REMOVE stub 'Rev. W. E. Costner, Falls Church'")

# #145: "Falls Church" -> append to Shiloh Baptist Church, Ordricks Corner
i = find_entry("Shiloh Baptist Church", "Virginia", "Northern Virginia",
               line_contains="Ordricks Corner")
if i is not None:
    church_entries[i]["lines"].append("Falls Church")
    print("  COLFIX Shiloh (Ordricks) <- 'Falls Church'")
j = find_oneliner("Falls Church", "Virginia", "Northern Virginia")
if j is not None:
    church_entries.pop(j); print("  REMOVE stub 'Falls Church'")

# #181/182: Springfield Baptist Church <- "Church Roads"
do_append("Springfield Baptist Church", ["Church Roads"],
          "Virginia", "Shiloh")
i = find_oneliner("Church Roads", "Virginia", "Shiloh")
if i is not None:
    church_entries.pop(i); print("  REMOVE stub 'Church Roads' (Springfield)")

# #187/188: Union Gospel Baptist Church <- "Church Roads"
do_append("Union Gospel Baptist Church", ["Church Roads"],
          "Virginia", "Shiloh")
i = find_oneliner("Church Roads", "Virginia", "Shiloh")
if i is not None:
    church_entries.pop(i); print("  REMOVE stub 'Church Roads' (Union Gospel)")

# #191: "Christ Church" -> append to Calvary Baptist Church (Southside Rapp.)
i = find_entry("Calvary Baptist Church", "Virginia", "Southside Rappahannock",
               line_contains="Middlesex County; Rev. J. E. Wright")
if i is not None:
    church_entries[i]["lines"].append("Christ Church")
    print("  COLFIX Calvary Baptist (Southside) <- 'Christ Church'")
j = find_oneliner("Christ Church", "Virginia", "Southside Rappahannock")
if j is not None:
    church_entries.pop(j); print("  REMOVE stub 'Christ Church'")

# #192/193: Grafton Baptist Church <- "Middlesex County" + pastor
i = find_oneliner("Grafton Baptist Church", "Virginia", "Southside Rappahannock")
if i is not None:
    church_entries[i]["lines"].extend(["Middlesex County", "Rev. J. E. Wright, Christ Church"])
    print("  COLFIX Grafton <- ['Middlesex County', 'Rev. J. E. Wright, Christ Church']")
j = find_oneliner("Rev. J. E. Wright, Christ Church", "Virginia", "Southside Rappahannock")
if j is not None:
    church_entries.pop(j); print("  REMOVE stub 'Rev. J. E. Wright, Christ Church'")

# #201/202/203: Marmora Baptist Church <- "Church Road; Rev. L. A. James" + "Church Road"
i = find_oneliner("Marmora Baptist Church", "Virginia", "Eastern Virginia")
if i is not None:
    church_entries[i]["lines"].extend(["Church Road; Rev. L. A. James", "Church Road"])
    print("  COLFIX Marmora <- ['Church Road; Rev. L. A. James', 'Church Road']")
for stub in ["Church Road; Rev. L. A. James", "Church Road"]:
    j = find_oneliner(stub, "Virginia", "Eastern Virginia")
    if j is not None:
        church_entries.pop(j); print(f"  REMOVE stub {stub!r}")

# ------------------------------------------------------------------
# D. RENAME: OCR typos (entries otherwise genuinely short)
# ------------------------------------------------------------------
do_rename("Parish Chapel, Forrest on", "Parish Chapel, Forreston",
          "Texas", "Galilee-Griggs")                                                # #106
do_rename("New Elan Baptist Church, I anexa", "New Elam Baptist Church, Ianexa",
          "Virginia", "Shiloh")                                                     # #175
do_rename("Feach Grove Baptist Church, Louisa", "Peach Grove Baptist Church, Louisa",
          "Virginia", "Shiloh")                                                     # #178

# ------------------------------------------------------------------
# E. APPEND: county lines stripped by Step 2 (Star of the East)
# ------------------------------------------------------------------
star_county = [
    ("Angel View Baptist Church", "New Kent County"),
    ("Bethany Baptist Church", "Henrico County"),
    ("Fair Oak Baptist Church", "Henrico County"),
    ("Gravel Hill Baptist Church", "Henrico County"),
    ("Mt. Olive Baptist Church", "New Kent County"),
    ("Mt. Sterling Baptist Church", "Charles City County"),
    ("Mt. Tabor Baptist Church", "Henrico County"),
    ("New Branch Baptist Church", "New Kent County"),
    ("New Bridge Baptist Church", "Henrico County"),
    ("Rising Mount Zion Baptist Church", "Henrico County"),
    ("St. James Baptist Church", "Henrico County"),
    ("St. John Baptist Church", "Charles City County"),
    ("Second Bethel Baptist Church", "Henrico County"),
    ("Second Elam Baptist Church", "New Kent County"),
    ("Seven Pines Baptist Church", "Henrico County"),
]
for name, county in star_county:
    do_append(name, [county], "Virginia", "Star of the East")
# Two Union Baptist Churches distinguished by county
unions = [(i, e) for i, e in enumerate(church_entries)
          if len(e["lines"]) == 1 and e["lines"][0] == "Union Baptist Church"
          and e["state"] == "Virginia"
          and "Star of the East" in (e["association"] or "")]
if len(unions) == 2:
    church_entries[unions[0][0]]["lines"].append("Charles City County")
    church_entries[unions[1][0]]["lines"].append("New Kent County")
    print("  APPEND Union Baptist Church (x2) <- counties")

# ------------------------------------------------------------------
# F. APPEND: county lines stripped (Southside Rappahannock)
# ------------------------------------------------------------------
do_append("Mt. Olive Baptist Church", ["Middlesex County"],
          "Virginia", "Southside Rappahannock")                                     # #194

# ------------------------------------------------------------------
# G. APPEND: D.C. pastor/address lines lost in two-column OCR
# ------------------------------------------------------------------
do_append("Carron Baptist Church", ["935 1/2 Florida Ave. N. W.",
          "Rev. Patrick Yancy", "1914 New Hampshire Ave. N. W."],
          "Washington", "General Baptist")                                          # #235
do_append("First New Hope Baptist Church", ["1337 7th St. N. W.",
          "Rev. Earnest Wilson, 1228 4th St., N.W."],
          "Washington", "General Baptist")                                          # #236

print(f"\nEntries: {before} -> {len(church_entries)}")
print("Remaining 1-line entries:",
      sum(1 for e in church_entries if len(e["lines"]) == 1))
print("Lines per entry:",
      dict(sorted(Counter(len(e["lines"]) for e in church_entries).items())))

  APPEND 'Maud Chapel, Maud' <- ['Rev. Tom Temple, Redwater']
  REMOVE stub 'Rev. Tom Temple, Redwater'
  COLFIX Pine Bluff <- ['Hughes Springs; Rev. Thomas Temple', 'Redwater']
  REMOVE stub 'Hughes Springs; Rev. Thomas Temple'
  COLFIX Eighth Street <- ['Temple; Rev. D. P. Prownell', '203 S. 10th Ave., Dallas']
  REMOVE stub 'Temple; Rev. D. P. Prownell'
  COLFIX Heidenheimer <- 'R.F.D. 3, Temple'
  REMOVE orphan 'R.F.D. 3, Temple'
  COLFIX Antioch Baptist (Heidenheimer) <- 'Temple'
  !! 2 matches for 1-liner 'Temple'
  REMOVE orphan 'Temple'
  APPEND 'Mt. Ozia Baptist Church, Rogers' <- ['Rev. J. W. Wesley, Temple']
  REMOVE stub 'Rev. J. W. Wesley, Temple'
  COLFIX Mt. Zion (St. John) <- ['Temple; Rev. A. G. Harris', 'Temple']
  REMOVE stub 'Temple; Rev. A. G. Harris'
  REMOVE stub 'Temple'
  APPEND 'Ebenezer Baptist Church' <- ['Chapel Hill: Rev. C. C. Reed']
  REMOVE stub 'Chapel Hill: Rev. C. C. Reed'
  APPEND 'Chantilly Baptist Church, Chantilly' <- ['Rev. Oliver Hall, Falls Ch

## Step 4: Parse Entries into Structured Columns

Each church entry (1–5 lines of text) is parsed into four columns:

| Column | Content | Source |
|--------|---------|--------|
| `church_name` | Church name only | First line, before city comma |
| `church_city` | City, county, or street address of the church | First line after comma, or lines before pastor |
| `pastor_name` | Pastor name with Rev. prefix | Line containing Rev. marker |
| `pastor_address` | Pastor's mailing address | Text after pastor name comma, plus subsequent lines |

Plus `state` and `association` from the grouping metadata.

### Parsing logic

1. **Find the Rev. marker** — detects OCR variants: `Rev.`, `Rov.`, `kev.`, `Rev,`, `Rey.`, `Re v.`, `ev.`, `ov.`, `Rcv.`, `Kcv.`, etc.
2. **Split at the Rev. marker** — everything before is church info; everything from Rev. onward is pastor info
3. **Handle semicolons** — `"City; Rev. Name"` → city goes to `church_city`, rest to pastor
4. **Handle commas before Rev.** — `"City, Rev. Name"` → same split as semicolons
5. **Parse church name** — split first line at comma after "Church"/"Chapel" keyword
6. **Parse pastor name** — split at first comma after the name to separate address

In [19]:
# =====================================================================
# Step 4 -- Parse each church entry into structured columns
# Input:  church_entries (list of dicts with 'lines', 'state', 'association')
# Output: DataFrame with church_name, church_city, pastor_name, pastor_address,
#         state, association
# =====================================================================

import pandas as pd

# --- 4.1 Rev. detector (covers all OCR variants) ---
REV_RE = re.compile(
    r'(?:'
    r'\b[RrKkBbLlHh][eEoOaAcC][vV][.,:]'
    r'|\bRe v\.'
    r'|\b[Rr]ey\.'
    r'|\bRe\.\s+[A-Z]'
    r'|\bev\.\s+[A-Z]'
    r'|\bov\.\s+[A-Z]'
    r')'
)

def find_rev(lines):
    for i, line in enumerate(lines):
        m = REV_RE.search(line)
        if m:
            return i, m.start()
    return None, None

def split_church_name_city(text):
    kw = re.search(r'\b(Church|Chapel|Tabernacle|Mission|Temple)\b', text)
    if kw:
        after = text[kw.end():]
        cp = after.find(',')
        if cp >= 0:
            pos = kw.end() + cp
            return text[:pos].strip(), text[pos+1:].strip()
        return text.strip(), ''
    last = text.rfind(',')
    if last >= 0:
        return text[:last].strip(), text[last+1:].strip()
    return text.strip(), ''

def split_pastor_name_addr(text):
    m = REV_RE.search(text)
    if not m:
        return text.strip(), ''
    rev_marker = text[m.start():m.end()].rstrip('.,: ')
    after = text[m.end():].strip()
    # Walk through commas; skip commas after single-letter initials (OCR period-as-comma)
    # and commas before Jr./Sr. suffixes
    pos = 0
    name_end = len(after)
    while pos < len(after):
        cp = after.find(',', pos)
        if cp < 0:
            break
        before_comma = after[:cp].strip()
        after_comma = after[cp+1:].strip()
        if re.search(r'\b[A-Z]$', before_comma):
            pos = cp + 1
            continue
        if re.match(r'[Jj]r\.?\b|[Ss]r\.?\b', after_comma):
            pos = cp + 1
            continue
        name_end = cp
        break
    if name_end < len(after):
        name_part = after[:name_end].strip()
        addr_part = after[name_end+1:].strip()
    else:
        name_part = after.strip()
        addr_part = ''
    full = rev_marker + '. ' + name_part
    return full.replace('..', '.').strip(), addr_part

def parse_entry(entry):
    lines = entry['lines']
    state = entry.get('state', '')
    assoc = entry.get('association', '') or ''
    rev_line, rev_pos = find_rev(lines)
    if rev_line is not None:
        before_rev = lines[rev_line][:rev_pos].strip()
        after_rev = lines[rev_line][rev_pos:].strip()
        church_lines = list(lines[:rev_line])
        if before_rev:
            church_lines.append(before_rev.rstrip(';, '))
        pastor_lines = [after_rev] + list(lines[rev_line+1:])
    else:
        church_lines = list(lines)
        pastor_lines = []
    church_name, church_city = '', ''
    if church_lines:
        church_name, city0 = split_church_name_city(church_lines[0])
        parts = [city0] if city0 else []
        for cl in church_lines[1:]:
            if cl.strip():
                parts.append(cl.strip())
        church_city = '; '.join(parts)
    pastor_name, pastor_addr = '', ''
    if pastor_lines:
        pastor_name, paddr0 = split_pastor_name_addr(pastor_lines[0])
        # If name is only initials (no surname), merge next line before splitting
        name_after_rev = re.sub(r'^(Rev|Rov|Rcv|Kcv|kev|Bev|Hev)\.\s*', '', pastor_name)
        tokens = [t for t in re.split(r'[.,\s]+', name_after_rev) if t]
        if len(pastor_lines) > 1 and all(len(t) <= 1 for t in tokens):
            merged = pastor_lines[0].rstrip() + ' ' + pastor_lines[1].lstrip()
            pastor_name, paddr0 = split_pastor_name_addr(merged)
            remaining = pastor_lines[2:]
        else:
            remaining = pastor_lines[1:]
        parts = [paddr0] if paddr0 else []
        for pl in remaining:
            if pl.strip():
                parts.append(pl.strip())
        pastor_addr = '; '.join(parts)
    return {
        'church_name': church_name,
        'church_city': church_city,
        'pastor_name': pastor_name,
        'pastor_address': pastor_addr,
        'state': state,
        'association': assoc,
    }

# --- Parse all entries ---
parsed = [parse_entry(e) for e in church_entries]
df = pd.DataFrame(parsed)

print(f'Parsed {len(df):,} entries into DataFrame')
print()
print('Column fill rates:')
for col in ['church_name', 'church_city', 'pastor_name', 'pastor_address']:
    filled = (df[col] != '').sum()
    print(f'  {col:16s}: {filled:5,} / {len(df):,}  ({100*filled/len(df):.1f}%)')

print()
print('--- Sample parsed entries ---')
for i in [0, 100, 500, 1000, 2000, 3000, 4000, 5000, 6000, 7000]:
    if i < len(df):
        r = df.iloc[i]
        print(f'  [{i}] {r["church_name"]} | {r["church_city"]} | {r["pastor_name"]} | {r["pastor_address"]}')

Parsed 7,355 entries into DataFrame

Column fill rates:
  church_name     : 7,355 / 7,355  (100.0%)
  church_city     : 6,756 / 7,355  (91.9%)
  pastor_name     : 6,843 / 7,355  (93.0%)
  pastor_address  : 5,941 / 7,355  (80.8%)

--- Sample parsed entries ---
  [0] Antioch Baptist Church | Natchez | Rev. C. R. Anderson | 10 Keim Ave., Natchez
  [100] Mt. Pleasant Baptist Church | Payden | Rev. E. E. Spencer | Lorman
  [500] Beach Grove Baptist Church | Vicksburg | Rev. Sam Joyce | Vicksburg
  [1000] Mt. Arratt Baptist Church | Tralake | Rev. J. D. Johnson | Greenville
  [2000] Big Ruin Creek Baptist Church | Henderson | Rev. J. W. Burwell | R. F. D. 1, Henderson
  [3000] St. Matthew Baptist Church | Berkeley County | Rev. I. Thomas | Russellville
  [4000] St. Paul Baptist Church | Hampshire |  | 
  [5000] Mt. Carmel Baptist Church | Paris |  | 
  [6000] First Baptist Church | Chilhowie | Rev. J. H. Hardy | Meadowview
  [7000] St. Bethel Baptist Church | 320 I St, S. E. | Rev. Adolphus 

### 4.1 Post-parse manual corrections

Fix parsing artifacts that the Step 4 parser cannot handle mechanically:
- OCR comma-as-period in pastor initials (fixed in parser: 47 entries)
- Line-split pastor names where `Rev. L.` / `W. Williams` spans two lines
- Double-punctuation `Rev, T., Smith` where the initial has both comma and period

In [20]:
# =====================================================================
# Section 4.1 -- Post-parse manual corrections
# Fix entries the parser cannot handle mechanically (heavy OCR garble,
# line-break edge cases, double-punctuation).
# =====================================================================

fixes = 0

# [4172] Orchard Knob: semicolon in merged name "Rev. H. J. Johnson; Chattanooga"
mask = (df['church_name'] == 'Orchard Knob Baptist Church') & (df['pastor_name'].str.contains('Johnson; Chattanooga', na=False))
if mask.sum() == 1:
    idx = mask.idxmax()
    df.at[idx, 'pastor_name'] = 'Rev. H. J. Johnson'
    df.at[idx, 'pastor_address'] = 'Chattanooga'
    print(f'  fix [{idx}] Orchard Knob: split semicolon in pastor name')
    fixes += 1

# [639] Mt. Herald: "Rev, i, F. Richardson" -> Rev. I. F. Richardson
mask = (df['church_name'].str.contains('Herald', na=False)) & (df['pastor_name'] == 'Rev. i')
if mask.sum() == 1:
    idx = mask.idxmax()
    df.at[idx, 'pastor_name'] = 'Rev. I. F. Richardson'
    df.at[idx, 'pastor_address'] = '113 Katzmaire St., Leland'
    print(f'  fix [{idx}] Mt. Herald: OCR garble -> Rev. I. F. Richardson')
    fixes += 1

# [1683] It. Morich (Mt. Moriah): "Rov. 3. 3, Givins" -> Rev. J. J. Givins (OCR 3->J)
mask = (df['church_name'].str.contains('Morich', na=False)) & (df['pastor_name'].str.contains('3', na=False))
if mask.sum() == 1:
    idx = mask.idxmax()
    df.at[idx, 'pastor_name'] = 'Rov. J. J. Givins'
    df.at[idx, 'pastor_address'] = '2008 5th Ave.; H.T.C.'
    print(f'  fix [{idx}] Mt. Moriah: OCR 3->J -> Rov. J. J. Givins')
    fixes += 1

# [3990] Limestone: "Rev. 0, E. Galloway" -> Rev. O. E. Galloway (OCR 0->O)
mask = (df['church_name'] == 'Limestone Baptist Church') & (df['pastor_name'] == 'Rev. 0')
if mask.sum() == 1:
    idx = mask.idxmax()
    df.at[idx, 'pastor_name'] = 'Rev. O. E. Galloway'
    df.at[idx, 'pastor_address'] = ''
    print(f'  fix [{idx}] Limestone: OCR 0->O -> Rev. O. E. Galloway')
    fixes += 1

# [5825] Macedonia: "Rev. u, A. Stokes" -> Rev. U. A. Stokes (OCR u->U)
mask = (df['church_name'] == 'Macedonia Baptist Church') & (df['pastor_name'] == 'Rev. u')
if mask.sum() == 1:
    idx = mask.idxmax()
    df.at[idx, 'pastor_name'] = 'Rev. U. A. Stokes'
    df.at[idx, 'pastor_address'] = ''
    print(f'  fix [{idx}] Macedonia: OCR u->U -> Rev. U. A. Stokes')
    fixes += 1

# [6551] Union Baptist Ashland: "Rev, T., Smith, Ashland" -> Rev. T. Smith
mask = (df['church_name'] == 'Union Baptist Church') & (df['church_city'] == 'Ashland') & (df['state'] == 'Virginia')
if mask.sum() == 1:
    idx = mask.idxmax()
    if 'Smith' in df.at[idx, 'pastor_address']:
        df.at[idx, 'pastor_name'] = 'Rev. T. Smith'
        df.at[idx, 'pastor_address'] = 'Ashland'
        print(f'  fix [{idx}] Union Baptist Ashland: -> Rev. T. Smith')
        fixes += 1

print(f'\nApplied {fixes} post-parse corrections')


  fix [4172] Orchard Knob: split semicolon in pastor name
  fix [639] Mt. Herald: OCR garble -> Rev. I. F. Richardson
  fix [1683] Mt. Moriah: OCR 3->J -> Rov. J. J. Givins
  fix [3990] Limestone: OCR 0->O -> Rev. O. E. Galloway
  fix [5825] Macedonia: OCR u->U -> Rev. U. A. Stokes
  fix [6551] Union Baptist Ashland: -> Rev. T. Smith

Applied 6 post-parse corrections


In [21]:
# --- Export to CSV ---
out_path = '1.data/1.cleaned/black_churches_vol2_parsed.csv'
df.to_csv(out_path, index=False)
print(f'Exported {len(df):,} rows to {out_path}')
print()
print(df.head(10).to_string())

Exported 7,355 rows to 1.data/1.cleaned/black_churches_vol2_parsed.csv

                          church_name  church_city          pastor_name         pastor_address        state                                  association
0              Antioch Baptist Church      Natchez  Rev. C. R. Anderson  10 Keim Ave., Natchez  Mississippi  Adams County Missionary Baptist Association
1               Beulah Baptist Church      Natchez      Rev. B. D. Sims                         Mississippi  Adams County Missionary Baptist Association
2  Bright Morning Star Baptist Church      Natchez      Rev. R. A. Mays                         Mississippi  Adams County Missionary Baptist Association
3          China Grove Baptist Church      Natchez  Rev. C. R. Anderson  10 Keim Ave., Natchez  Mississippi  Adams County Missionary Baptist Association
4           Clay Mount Baptist Church      Natchez   Rev. J. A. Briscoe                         Mississippi  Adams County Missionary Baptist Association
5         

## Step 5: Geocode Churches (church_city + state → lat/lon)

Follows the same strategy as Volume 1 (`vol1_geocode_black_churches.py`):

1. For each **unique (church_city, state)** pair, try the **US Census one-line geocoder** first (no rate limit).
2. If Census returns nothing, fall back to **OpenStreetMap / Nominatim** (rate-limited to 1 req/sec).
3. If `church_city` is blank, fall back to `pastor_address` as the city string.
4. Results are **cached on disk** (`vol2_geocode_cache.json`) so the cell is fully resumable — interrupt and re-run to continue.
5. Output: `black_churches_vol2_geocoded.csv` with 5 new columns: `latitude`, `longitude`, `geocode_source`, `match_quality`, `geocode_query`.

`match_quality` is one of: `city` | `county` | `state` | `none`.


In [ ]:
# =====================================================================
# Step 5 -- Geocode church_city + state to lat/lon
# Same strategy as vol1_geocode_black_churches.py:
#   Census geocoder first, Nominatim fallback, disk cache for resume.
# =====================================================================
import json, time, urllib.parse, urllib.request
from pathlib import Path
from IPython.display import clear_output

CLEANED = Path('1.data/1.cleaned')
CACHE_PATH = CLEANED / 'vol2_geocode_cache.json'
OUT_CSV = CLEANED / 'black_churches_vol2_geocoded.csv'

CONTACT = 'leahziqian.liu33@outlook.com'
NOMINATIM_DELAY = 1.1

# --- helpers ---
def http_get_json(url, headers=None, timeout=30):
    req = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.load(resp)

def geocode_census(city, state):
    params = urllib.parse.urlencode({'address': f'{city}, {state}',
                                     'benchmark': 'Public_AR_Current', 'format': 'json'})
    url = 'https://geocoding.geo.census.gov/geocoder/locations/onelineaddress?' + params
    try:
        data = http_get_json(url, headers={'User-Agent': CONTACT})
    except Exception:
        return None
    matches = data.get('result', {}).get('addressMatches', [])
    if not matches:
        return None
    c = matches[0]['coordinates']
    return float(c['y']), float(c['x']), 'city'

def geocode_nominatim(city, state):
    params = urllib.parse.urlencode({'city': city, 'state': state, 'country': 'USA',
                                     'format': 'jsonv2', 'limit': 1, 'addressdetails': 1})
    url = 'https://nominatim.openstreetmap.org/search?' + params
    try:
        data = http_get_json(url, headers={'User-Agent': f'black-churches-geocoder ({CONTACT})'})
    except Exception:
        return None
    if not data:
        return None
    hit = data[0]
    lat, lon = float(hit['lat']), float(hit['lon'])
    addrtype = hit.get('addresstype', '')
    if addrtype in ('city','town','village','hamlet','municipality','suburb','neighbourhood','locality'):
        quality = 'city'
    elif addrtype == 'county':
        quality = 'county'
    elif addrtype == 'state':
        quality = 'state'
    else:
        quality = 'city'
    return lat, lon, quality

def geocode_one(city, state, cache):
    key = f'{city.strip().lower()}|{state.strip().lower()}'
    if key in cache:
        return cache[key]
    result = None
    res = geocode_census(city, state)
    if res:
        lat, lon, qual = res
        result = {'lat': lat, 'lon': lon, 'source': 'census',
                  'quality': qual, 'query': f'{city}, {state}'}
    else:
        time.sleep(NOMINATIM_DELAY)
        res = geocode_nominatim(city, state)
        if res:
            lat, lon, qual = res
            result = {'lat': lat, 'lon': lon, 'source': 'nominatim',
                      'quality': qual, 'query': f'{city}, {state}'}
    if result is None:
        result = {'lat': '', 'lon': '', 'source': 'none',
                  'quality': 'none', 'query': f'{city}, {state}'}
    cache[key] = result
    return result

def save_cache(cache):
    tmp = str(CACHE_PATH) + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=0)
    import os; os.replace(tmp, str(CACHE_PATH))

# --- load cache ---
if CACHE_PATH.exists():
    cache = json.loads(CACHE_PATH.read_text())
    print(f'Loaded cache: {len(cache)} entries')
else:
    cache = {}
    print('Starting with empty cache')

# --- build unique (city, state) pairs ---
def loc_for(row):
    city = str(row.get('church_city', '')).strip()
    if not city or city == 'nan':
        city = str(row.get('pastor_address', '')).strip()
    state = str(row.get('state', '')).strip()
    if city == 'nan': city = ''
    if state == 'nan': state = ''
    return city, state

uniq = {}
for _, row in df.iterrows():
    city, state = loc_for(row)
    if city and state:
        uniq[(city.lower(), state.lower())] = (city, state)

total = len(uniq)
already = sum(1 for c, s in uniq.keys() if f'{c}|{s}' in cache)
print(f'{len(df):,} churches, {total:,} unique (city, state) pairs')
print(f'Already cached: {already:,}, remaining: {total - already:,}')

# --- geocode loop (resumable) ---
done = 0
new_this_run = 0
save_every = 25
t0 = time.time()

for city, state in uniq.values():
    key = f'{city.strip().lower()}|{state.strip().lower()}'
    was_cached = key in cache
    geocode_one(city, state, cache)
    done += 1
    if not was_cached:
        new_this_run += 1
        if new_this_run % save_every == 0:
            save_cache(cache)
    if done % 100 == 0 or done == total:
        elapsed = time.time() - t0
        hits = sum(1 for v in cache.values() if v['source'] != 'none')
        clear_output(wait=True)
        print(f'Progress: {done:,}/{total:,} ({100*done/total:.1f}%)  |  '
              f'new this run: {new_this_run}  |  cache hits: {hits}  |  '
              f'elapsed: {elapsed:.0f}s')

save_cache(cache)
print('\nGeocoding complete. Cache saved.')

# --- join results back to df and export ---
lats, lons, sources, qualities, queries = [], [], [], [], []
for _, row in df.iterrows():
    city, state = loc_for(row)
    key = f'{city.strip().lower()}|{state.strip().lower()}'
    g = cache.get(key)
    if g:
        lats.append(g['lat']); lons.append(g['lon'])
        sources.append(g['source']); qualities.append(g['quality'])
        queries.append(g['query'])
    else:
        lats.append(''); lons.append('')
        sources.append('none'); qualities.append('none')
        queries.append('')

df['latitude'] = lats
df['longitude'] = lons
df['geocode_source'] = sources
df['match_quality'] = qualities
df['geocode_query'] = queries

# replace empty strings with NaN for numeric columns
import numpy as np
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

df.to_csv(OUT_CSV, index=False)

# --- summary ---
matched = df['geocode_source'].ne('none').sum()
print(f'\nWrote: {OUT_CSV}')
print(f'Churches geocoded: {matched:,}/{len(df):,} ({100*matched/len(df):.1f}%)')
print(f'\nBy match_quality:')
print(df['match_quality'].value_counts().to_string())
print(f'\nBy geocode_source:')
print(df['geocode_source'].value_counts().to_string())

### 5.1 Recover Unmatched Churches

Follows `vol1_recover_unmatched_churches.py`. For each unmatched row:

1. Clean `church_city`: strip R.F.D./Box/route cruft, split embedded state suffixes (e.g. ", Tenn.").
2. Generate OCR spelling variants (reverse e→o, e→c errors; rejoin split tokens).
3. Re-geocode via Nominatim with each variant.
4. **Guardrail**: accept a hit only if the point falls inside the expected state (point-in-polygon against Census state shapefile).
5. Use `pastor_address` only when it's an OCR-variant of the same town (never a different town).


In [ ]:
# =====================================================================
# Step 5.1 -- Recover unmatched churches
# Clean city names, generate OCR variants, re-geocode with state guardrail.
# =====================================================================
import json, re
from pathlib import Path
import pandas as pd
import shapefile
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

CLEANED = Path('1.data/1.cleaned')
DATA = Path('1.data')
IN_CSV = CLEANED / 'black_churches_vol2_geocoded.csv'
OUT_CSV = CLEANED / 'black_churches_vol2_geocoded_recovered.csv'
LOG_CSV = CLEANED / 'black_churches_vol2_recovery_log.csv'
CACHE = CLEANED / '.vol2_recovery_geocode_cache.json'
STATE_SHP = DATA / '2.raw/Shapefiles/US States/cb_2018_us_state_20m/cb_2018_us_state_20m.shp'

# --- State-boundary validator (ray-casting point-in-polygon) ---
def _load_states(shp_path):
    r = shapefile.Reader(str(shp_path))
    fields = [f[0] for f in r.fields[1:]]
    name_i = fields.index('NAME')
    out = []
    for sr in r.shapeRecords():
        sh = sr.shape
        parts = list(sh.parts) + [len(sh.points)]
        rings = [sh.points[parts[i]:parts[i+1]] for i in range(len(parts)-1)]
        out.append((sr.record[name_i], sh.bbox, rings))
    return out

_STATES = _load_states(STATE_SHP)
print(f'Loaded {len(_STATES)} state polygons')

def _in_ring(x, y, ring):
    inside = False
    n = len(ring)
    j = n - 1
    for i in range(n):
        xi, yi = ring[i]
        xj, yj = ring[j]
        if ((yi > y) != (yj > y)) and (x < (xj - xi) * (y - yi) / (yj - yi) + xi):
            inside = not inside
        j = i
    return inside

def state_of(lon, lat):
    for name, (xmin, ymin, xmax, ymax), rings in _STATES:
        if not (xmin <= lon <= xmax and ymin <= lat <= ymax):
            continue
        inside = False
        for ring in rings:
            if _in_ring(lon, lat, ring):
                inside = not inside
        if inside:
            return name
    return None

# --- Candidate generation ---
ABBR2STATE = {
    'ala': 'Alabama', 'ariz': 'Arizona', 'ark': 'Arkansas',
    'cal': 'California', 'calif': 'California', 'colo': 'Colorado',
    'conn': 'Connecticut', 'del': 'Delaware', 'fla': 'Florida',
    'ga': 'Georgia', 'ill': 'Illinois', 'ind': 'Indiana',
    'iowa': 'Iowa', 'kan': 'Kansas', 'kans': 'Kansas', 'ky': 'Kentucky',
    'la': 'Louisiana', 'mass': 'Massachusetts', 'md': 'Maryland',
    'mich': 'Michigan', 'minn': 'Minnesota', 'miss': 'Mississippi',
    'mo': 'Missouri', 'mont': 'Montana', 'neb': 'Nebraska',
    'nebr': 'Nebraska', 'nev': 'Nevada', 'nc': 'North Carolina',
    'nd': 'North Dakota', 'ohio': 'Ohio', 'okla': 'Oklahoma',
    'ore': 'Oregon', 'pa': 'Pennsylvania', 'sc': 'South Carolina',
    'sd': 'South Dakota', 'tenn': 'Tennessee', 'tex': 'Texas',
    'texas': 'Texas', 'va': 'Virginia', 'vt': 'Vermont',
    'wash': 'Washington', 'wis': 'Wisconsin', 'wisc': 'Wisconsin',
    'wva': 'West Virginia', 'wyo': 'Wyoming',
}
_ADDR_PREFIX = re.compile(r"(?i)\b(r\.?\s*f\.?\s*d\.?|box|route|rt|p\.?\s*o\.?)\b[\s\.]*\d*")
_NUM_TAG = re.compile(r'(?i)\bno\.?\s*\d+\b')
_STREET_RE = re.compile(r'(?i)^\s*\d+\s.*\b(st|ave|rd|pl|blvd|street|avenue|road|place)\b\.?')

def _clean_core(s):
    s = _NUM_TAG.sub('', s)
    s = _ADDR_PREFIX.sub('', s)
    s = s.replace(';', ' ').replace('\u00b7', ' ')
    s = re.sub(r'\s+', ' ', s).strip(' ,.-;')
    return s

def _ocr_variants(core):
    out = [core]
    nospace = core.replace(' ', '')
    if nospace != core:
        out.append(nospace)
    for src in (core, nospace):
        out.append(src.replace('o', 'e'))
        out.append(src.replace('c', 'e'))
        out.append(src.replace('o', 'e').replace('c', 'e'))
        for ch in ('o', 'c'):
            idx = -1
            while True:
                idx = src.find(ch, idx + 1)
                if idx < 0: break
                out.append(src[:idx] + 'e' + src[idx+1:])
    seen, uniq = set(), []
    for v in out:
        v = v.strip()
        if len(v) >= 3 and v.lower() not in seen:
            seen.add(v.lower()); uniq.append(v)
    return uniq[:14]

def _fuzzy_same(a, b):
    a, b = a.lower(), b.lower()
    if not a or not b: return False
    if a == b or a in b or b in a: return True
    sa, sb = set(a.replace(' ', '')), set(b.replace(' ', ''))
    return len(sa & sb) / max(1, len(sa | sb)) >= 0.8 and abs(len(a) - len(b)) <= 3

def build_candidates(church_city, pastor_address, listed_state):
    cands = []
    cc = '' if pd.isna(church_city) else str(church_city).strip()
    pa = '' if pd.isna(pastor_address) else str(pastor_address).strip()
    if cc.lower() in ('', 'nan'): return cands
    if _STREET_RE.match(cc):
        cc = re.split(r'[;,]', cc)[-1].strip() or cc
    m = re.search(r',\s*([A-Za-z]{2,6})\.?\s*$', cc)
    embedded_state = None; core = cc
    if m and m.group(1).lower() in ABBR2STATE:
        embedded_state = ABBR2STATE[m.group(1).lower()]
        core = cc[:m.start()]
    core = _clean_core(core)
    target = embedded_state or listed_state
    for v in _ocr_variants(core):
        cands.append((f'{v}, {target}', target))
    pa_core = _clean_core(re.sub(r',\s*[A-Za-z]{2,6}\.?\s*$', '', pa))
    if pa_core and _fuzzy_same(pa_core, core):
        for v in _ocr_variants(pa_core):
            q = f'{v}, {target}'
            if q not in [c[0] for c in cands]:
                cands.append((q, target))
    seen, out = set(), []
    for q, st in cands:
        if q.lower() not in seen:
            seen.add(q.lower()); out.append((q, st))
    return out

# --- Geocoder (cached, rate-limited) ---
_geolocator = Nominatim(user_agent='leahziqian.liu33@outlook.com', timeout=60)
_geocode = RateLimiter(_geolocator.geocode, min_delay_seconds=1)

_cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
print(f'Recovery cache: {len(_cache)} entries')

def geocode_us(query):
    if query in _cache:
        v = _cache[query]
        return tuple(v) if v else None
    loc = _geocode(query, country_codes='us', exactly_one=True)
    res = [loc.latitude, loc.longitude] if loc else None
    _cache[query] = res
    if len(_cache) % 50 == 0:
        CACHE.write_text(json.dumps(_cache))
    return tuple(res) if res else None

# --- Run recovery ---
df = pd.read_csv(IN_CSV)
mask = df['match_quality'].eq('none')
print(f'Unmatched rows to attempt: {int(mask.sum())}')

import sys
recovered = 0; xstate = 0; log_rows = []
total_unmatched = int(mask.sum())
processed = 0

for i in df.index[mask]:
    row = df.loc[i]
    processed += 1
    cands = build_candidates(row['church_city'], row['pastor_address'], row['state'])
    for query, exp_state in cands:
        hit = geocode_us(query)
        if not hit: continue
        lat, lon = hit
        if state_of(lon, lat) != exp_state: continue
        df.at[i, 'latitude'] = lat
        df.at[i, 'longitude'] = lon
        df.at[i, 'match_quality'] = 'city'
        cross = exp_state != row['state']
        df.at[i, 'geocode_source'] = 'nominatim_recovered_xstate' if cross else 'nominatim_recovered'
        df.at[i, 'geocode_query'] = query
        recovered += 1; xstate += int(cross)
        log_rows.append({'index': i, 'church_name': row['church_name'],
                         'church_city_raw': row['church_city'],
                         'pastor_address': row['pastor_address'],
                         'listed_state': row['state'], 'accepted_query': query,
                         'geocoded_state': exp_state, 'latitude': lat,
                         'longitude': lon, 'cross_state': cross})
        break
    if processed % 100 == 0:
        print(f'  processed {processed}/{total_unmatched}, recovered {recovered} so far')
        sys.stdout.flush()

CACHE.write_text(json.dumps(_cache))

df.to_csv(OUT_CSV, index=False)
pd.DataFrame(log_rows).to_csv(LOG_CSV, index=False)

still_none = int(df['match_quality'].eq('none').sum())
print(f'\n--- Recovery results ---')
print(f'recovered           : {recovered}')
print(f'  of which x-state  : {xstate}')
print(f'still unmatched     : {still_none}')
print(f'overall match rate  : {100*(df["match_quality"] != "none").mean():.1f}%')
print(f'\nwrote: {OUT_CSV.name}')
print(f'       {LOG_CSV.name}')

### 5.2 Match Churches to 1980 SMSAs

Follows `vol1_match_churches_to_smsa.py`. For each geocoded church:

1. Reproject lat/lon (EPSG:4326) into the NHGIS SMSA shapefile's Albers CRS.
2. Point-in-polygon test against all 1980 SMSA polygons.
3. Assign `smsaa` (SMSA code), `smsa_name`, `smsa_gisjoin`.
4. Non-metropolitan churches get blank SMSA fields.


In [ ]:
# =====================================================================
# Step 5.2 -- Match churches to 1980 SMSAs by spatial overlay
# =====================================================================
import shapefile
from pyproj import Transformer
from pathlib import Path
import pandas as pd

DATA = Path('1.data')
CLEANED = DATA / '1.cleaned'
IN_CSV = CLEANED / 'black_churches_vol2_geocoded_recovered.csv'
OUT_CSV = CLEANED / 'black_churches_vol2_smsa.csv'
SMSA_SHP = (DATA / '2.raw/Shapefiles/smsa/nhgis0006_shape'
            / 'nhgis0006_shapefile_tl2000_us_smsa_1980/US_smsa_1980.shp')

# --- Load SMSA polygons ---
r = shapefile.Reader(str(SMSA_SHP))
fields = [f[0] for f in r.fields[1:]]
i_code = fields.index('SMSAA')
i_name = fields.index('SMSA')
i_gis = fields.index('GISJOIN')

smsas = []
for sr in r.shapeRecords():
    sh = sr.shape
    parts = list(sh.parts) + [len(sh.points)]
    rings = [sh.points[parts[k]:parts[k+1]] for k in range(len(parts)-1)]
    smsas.append((sr.record[i_code], sr.record[i_name], sr.record[i_gis],
                  sh.bbox, rings))
print(f'SMSA polygons loaded: {len(smsas)}')

_wkt = Path(str(SMSA_SHP).replace('.shp', '.prj')).read_text()
_to_albers = Transformer.from_crs('EPSG:4326', _wkt, always_xy=True)

# --- Point-in-polygon ---
def _in_ring(x, y, ring):
    inside = False
    n = len(ring); j = n - 1
    for k in range(n):
        xi, yi = ring[k]; xj, yj = ring[j]
        if ((yi > y) != (yj > y)) and (x < (xj - xi) * (y - yi) / (yj - yi) + xi):
            inside = not inside
        j = k
    return inside

def smsa_of(lon, lat):
    x, y = _to_albers.transform(lon, lat)
    for code, name, gis, (xmin, ymin, xmax, ymax), rings in smsas:
        if not (xmin <= x <= xmax and ymin <= y <= ymax): continue
        inside = False
        for ring in rings:
            if _in_ring(x, y, ring): inside = not inside
        if inside:
            return code, name, gis
    return None, None, None

# --- Overlay ---
df = pd.read_csv(IN_CSV)
has_xy = df['latitude'].notna() & df['longitude'].notna()
print(f'churches with coordinates: {int(has_xy.sum())} / {len(df)}')

codes, names, gids = [], [], []
for lat, lon, ok in zip(df['latitude'], df['longitude'], has_xy):
    if not ok:
        codes.append(None); names.append(None); gids.append(None)
        continue
    c, n, g = smsa_of(lon, lat)
    codes.append(c); names.append(n); gids.append(g)

df['smsaa'] = codes
df['smsa_name'] = names
df['smsa_gisjoin'] = gids

df.to_csv(OUT_CSV, index=False)

in_smsa = df['smsaa'].notna()
print(f'\n--- SMSA matching results ---')
print(f'in an SMSA (metropolitan) : {int(in_smsa.sum())} '
      f'({100 * in_smsa.sum() / max(1, has_xy.sum()):.1f}% of geocoded)')
print(f'non-metropolitan          : {int((has_xy & ~in_smsa).sum())}')
print(f'distinct SMSAs hit        : {df["smsaa"].nunique()}')
print(f'\ntop 12 SMSAs by church count:')
print(df.loc[in_smsa, 'smsa_name'].value_counts().head(12).to_string())
print(f'\nwrote: {OUT_CSV}')